# 25-Day PySpark Learning Plan

**Beginner → advanced | Data Engineer • Azure Data Engineer • PySpark Developer**

This is a complete classroom notebook built around two continuous projects:

- **Instructor project:** Retail Sales Analytics Platform
- **Student project:** Banking Transaction Analytics Platform

Every day follows: **Concept → Notes → Syntax → Retail Example → Explanation → Banking Assignment → Expected Result → Interview Questions**.

> Teaching balance: approximately 30% theory and 70% hands-on practice. Complete the notebook in order because later days intentionally reuse earlier concepts.

## How to use this notebook

1. Study one numbered day per session (about 2–3 hours).
2. Run the **Course Lab Setup** once at the start of a fresh kernel.
3. Predict every schema and result before running its code cell.
4. Complete the ten retail practice questions, then the banking assignment without looking for a full solution.
5. Keep a separate answer notebook or Git repository for your work.
6. End each session by speaking the interview answers aloud in your own words.

**Local prerequisites:** Java 8/11/17 (compatible with your PySpark release), Python, Jupyter, and PySpark. In managed Azure Databricks or Microsoft Fabric, use the provided Spark runtime instead of creating a local cluster.

## Environment notes

- The notebook sets Spark to `local[2]` for a laptop-friendly classroom session. Remove the local master setting when a managed platform supplies the cluster.
- On native Windows, in-memory transformations work with a normal PySpark install, but local CSV/JSON/Parquet **writes** can require a compatible Hadoop Windows setup (`HADOOP_HOME` and `winutils.exe`). A managed Spark service, WSL/Linux environment, or an organization-approved Hadoop installation avoids that local filesystem limitation.
- If a Spark session already exists, `getOrCreate()` reuses it. Restart the kernel after changing Spark packages or Delta extensions.
- Day 23 requires a `delta-spark` version compatible with the active Spark runtime. Databricks runtimes usually provide Delta already.

In [ ]:
# Optional local installation — uncomment only if your environment needs it.
# %pip install pyspark
# Day 23 only (choose a delta-spark version compatible with your PySpark runtime):
# %pip install delta-spark

## Curriculum map

| Phase | Days | Outcome |
|---|---:|---|
| Foundations | 1–4 | Understand Spark, architecture, DataFrames, and data sources |
| Core transformations | 5–9 | Select, filter, transform, clean strings, and work with dates |
| Analytics | 10–16 | Aggregate, group, join, combine, handle nulls, and use windows |
| Execution and engineering | 17–22 | Use UDFs/RDDs carefully, reason about DAGs, partition, optimize, and write SQL |
| Lakehouse and projects | 23–25 | Use Delta concepts, build Bronze/Silver/Gold ETL, and finish both capstones |

## Course Lab Setup

The cell below creates realistic, small in-memory datasets. Its purpose is repeatability: all learners see the same results without downloading files. Production systems would read governed storage paths and secrets from environment-specific configuration.

The retail tables contain roughly 10–20 records where appropriate; the banking tables are independent assignment data. Monetary values use `double` for classroom simplicity—production financial systems normally use a suitable fixed-precision `DecimalType`.

In [ ]:
import os
import sys
from datetime import date
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType
)

# On Windows, Spark workers must use the same real Python executable as this kernel.
# This also avoids the Microsoft Store `python3.exe` application alias.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (SparkSession.builder
         .appName("25-Day PySpark Learning Plan")
         .master("local[2]")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.python.worker.reuse", "true")
         .config("spark.pyspark.python", sys.executable)
         .config("spark.pyspark.driver.python", sys.executable)
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

def make_df(rows, schema):
    """Create a small classroom DataFrame with an explicit schema."""
    return spark.createDataFrame(rows, schema)

# ---------------------------- Retail domain ----------------------------
retail_customers = make_df([
    ("C001", "Aarav Sharma", "Bengaluru", "KA", date(2024, 1, 15)),
    ("C002", "Diya Patel", "Mumbai", "MH", date(2024, 2, 10)),
    ("C003", "Kabir Singh", "Delhi", "DL", date(2024, 3, 12)),
    ("C004", "Meera Nair", "Kochi", "KL", date(2024, 4, 8)),
    ("C005", "Rohan Das", "Kolkata", "WB", date(2024, 5, 19)),
    ("C006", "Ananya Iyer", "Chennai", "TN", date(2024, 6, 22)),
    ("C007", "Vivaan Rao", "Hyderabad", "TS", date(2024, 7, 11)),
    ("C008", "Ishita Gupta", "Pune", "MH", date(2024, 8, 4)),
    ("C009", "Arjun Verma", "Jaipur", "RJ", date(2024, 9, 17)),
    ("C010", "Sara Khan", "Lucknow", "UP", date(2024, 10, 29)),
    ("C011", "Neel Joshi", "Ahmedabad", "GJ", date(2024, 11, 13)),
    ("C012", "Tara Menon", "Bengaluru", "KA", date(2025, 1, 5)),
], "customer_id string, customer_name string, city string, state string, signup_date date")

retail_products = make_df([
    ("P001", "Laptop Pro", "Electronics", 65000.0),
    ("P002", "Smartphone X", "Electronics", 28500.0),
    ("P003", "Wireless Earbuds", "Electronics", 2499.0),
    ("P004", "Office Chair", "Furniture", 8500.0),
    ("P005", "Study Desk", "Furniture", 12000.0),
    ("P006", "Running Shoes", "Fashion", 3999.0),
    ("P007", "Winter Jacket", "Fashion", 5499.0),
    ("P008", "Mixer Grinder", "Home", 4200.0),
    ("P009", "Cookware Set", "Home", 3299.0),
    ("P010", "Yoga Mat", "Sports", 999.0),
    ("P011", "Cricket Bat", "Sports", 2999.0),
    ("P012", "Coffee Pack", "Grocery", 349.0),
], "product_id string, product_name string, category string, price double")

retail_orders = make_df([
    ("O1001", "C001", "S001", date(2026, 1, 3), "COMPLETE", 2499.0),
    ("O1002", "C002", "S002", date(2026, 1, 5), "CANCELLED", 8500.0),
    ("O1003", "C003", "S004", date(2026, 1, 8), "COMPLETE", 3299.0),
    ("O1004", "C004", "S002", date(2026, 1, 14), "COMPLETE", 28500.0),
    ("O1005", "C005", "S003", date(2026, 1, 21), "COMPLETE", 5499.0),
    ("O1006", "C006", "S003", date(2026, 1, 25), "PENDING", 4200.0),
    ("O1007", "C007", "S001", date(2026, 2, 2), "COMPLETE", 12499.0),
    ("O1008", "C008", "S001", date(2026, 2, 6), "COMPLETE", 18500.0),
    ("O1009", "C009", "S003", date(2026, 2, 9), "COMPLETE", 1102.0),
    ("O1010", "C010", "S004", date(2026, 2, 13), "COMPLETE", 4200.0),
    ("O1011", "C011", "S002", date(2026, 2, 18), "COMPLETE", 12999.5),
    ("O1012", "C012", "S001", date(2026, 2, 20), "COMPLETE", 2150.5),
    ("O1013", "C001", "S001", date(2026, 2, 22), "COMPLETE", 65000.0),
    ("O1014", "C003", "S004", date(2026, 2, 24), "COMPLETE", 3000.0),
    ("O1015", "C007", "S003", date(2026, 2, 26), "COMPLETE", 1000.0),
], "order_id string, customer_id string, store_id string, order_date date, order_status string, total_amount double")

retail_order_items = make_df([
    ("O1001", "P003", 1, 2499.0, 0.00),
    ("O1002", "P004", 1, 8500.0, 0.00),
    ("O1003", "P009", 1, 3299.0, 0.00),
    ("O1004", "P002", 1, 28500.0, 0.00),
    ("O1005", "P007", 1, 5499.0, 0.00),
    ("O1006", "P008", 1, 4200.0, 0.00),
    ("O1007", "P005", 1, 12000.0, 0.00),
    ("O1007", "P012", 1, 499.0, 0.00),
    ("O1008", "P004", 2, 8500.0, 0.00),
    ("O1008", "P010", 2, 1000.0, 0.25),
    ("O1009", "P012", 2, 349.0, 0.00),
    ("O1009", "P010", 1, 449.0, 0.10),
    ("O1010", "P008", 1, 4200.0, 0.00),
    ("O1011", "P006", 2, 3999.0, 0.00),
    ("O1011", "P007", 1, 5499.0, 0.00),
    ("O1012", "P011", 1, 2999.0, 0.30),
    ("O1012", "P012", 1, 349.0, 0.85),
    ("O1013", "P001", 1, 65000.0, 0.00),
    ("O1014", "P003", 1, 2499.0, 0.00),
    ("O1014", "P012", 2, 349.0, 0.28),
    ("O1015", "P010", 1, 999.0, 0.00),
], "order_id string, product_id string, quantity int, unit_price double, discount double")

retail_stores = make_df([
    ("S001", "Bengaluru Central", "Bengaluru", "KA"),
    ("S002", "Mumbai High Street", "Mumbai", "MH"),
    ("S003", "Delhi Market", "Delhi", "DL"),
    ("S004", "Online Store", "Online", "ONLINE"),
], "store_id string, store_name string, city string, state string")

retail_payments = make_df([
    ("PAY01", "O1001", "UPI", 2499.0, "SUCCESS"),
    ("PAY02", "O1002", "CARD", 8500.0, "REFUNDED"),
    ("PAY03", "O1003", "UPI", 3299.0, "SUCCESS"),
    ("PAY04", "O1004", "CARD", 28500.0, "SUCCESS"),
    ("PAY05", "O1005", "CASH", 5499.0, "SUCCESS"),
    ("PAY06", "O1006", "UPI", 4200.0, "PENDING"),
    ("PAY07", "O1007", "CARD", 12499.0, "SUCCESS"),
    ("PAY08", "O1008", "CARD", 18500.0, "SUCCESS"),
    ("PAY09", "O1009", "UPI", 1102.0, "SUCCESS"),
    ("PAY10", "O1010", "UPI", 4200.0, "SUCCESS"),
    ("PAY11", "O1011", "CARD", 12999.5, "SUCCESS"),
    ("PAY12", "O1012", "CASH", 2150.5, "SUCCESS"),
], "payment_id string, order_id string, payment_type string, payment_amount double, payment_status string")

retail_inventory = make_df([
    ("P001", "S001", 8, date(2026, 2, 28)), ("P002", "S001", 14, date(2026, 2, 28)),
    ("P003", "S001", 40, date(2026, 2, 28)), ("P004", "S002", 6, date(2026, 2, 28)),
    ("P005", "S002", 5, date(2026, 2, 28)), ("P006", "S003", 22, date(2026, 2, 28)),
    ("P007", "S003", 11, date(2026, 2, 28)), ("P008", "S004", 35, date(2026, 2, 28)),
    ("P009", "S004", 30, date(2026, 2, 28)), ("P010", "S001", 55, date(2026, 2, 28)),
    ("P011", "S003", 17, date(2026, 2, 28)), ("P012", "S004", 120, date(2026, 2, 28)),
], "product_id string, store_id string, stock_quantity int, last_updated date")

# ---------------------------- Banking domain ---------------------------
bank_customers = make_df([
    ("BC001", "Aditya Mehta", "Mumbai", "RETAIL", date(2022, 1, 15)),
    ("BC002", "Priya Rao", "Bengaluru", "RETAIL", date(2022, 4, 10)),
    ("BC003", "Nikhil Shah", "Ahmedabad", "PREMIUM", date(2021, 7, 21)),
    ("BC004", "Fatima Ali", "Hyderabad", "RETAIL", date(2023, 2, 2)),
    ("BC005", "Karan Malhotra", "Delhi", "CORPORATE", date(2020, 11, 18)),
    ("BC006", "Lakshmi Nair", "Kochi", "PREMIUM", date(2021, 9, 9)),
    ("BC007", "Rahul Sen", "Kolkata", "RETAIL", date(2024, 1, 4)),
    ("BC008", "Zoya Khan", "Lucknow", "RETAIL", date(2023, 6, 12)),
    ("BC009", "Manav Joshi", "Pune", "CORPORATE", date(2020, 5, 20)),
    ("BC010", "Ira Kapoor", "Jaipur", "PREMIUM", date(2022, 8, 30)),
], "customer_id string, customer_name string, city string, customer_type string, join_date date")

bank_accounts = make_df([
    ("A001", "BC001", "SAVINGS", 125000.0, "B001"),
    ("A002", "BC002", "SAVINGS", 82000.0, "B002"),
    ("A003", "BC003", "CURRENT", 510000.0, "B003"),
    ("A004", "BC004", "SAVINGS", 45000.0, "B004"),
    ("A005", "BC005", "CURRENT", 1250000.0, "B005"),
    ("A006", "BC006", "SAVINGS", 230000.0, "B006"),
    ("A007", "BC007", "SAVINGS", 18000.0, "B007"),
    ("A008", "BC008", "SAVINGS", 67000.0, "B008"),
    ("A009", "BC009", "CURRENT", 890000.0, "B009"),
    ("A010", "BC010", "SAVINGS", 175000.0, "B010"),
    ("A011", "BC001", "CURRENT", 340000.0, "B001"),
    ("A012", "BC003", "SAVINGS", 92000.0, "B003"),
], "account_id string, customer_id string, account_type string, balance double, branch_id string")

bank_transactions = make_df([
    ("T001", "A001", date(2026, 1, 3), "UPI_DEBIT", 4500.0, "SUCCESS"),
    ("T002", "A002", date(2026, 1, 5), "NEFT_CREDIT", 75000.0, "SUCCESS"),
    ("T003", "A003", date(2026, 1, 8), "RTGS_DEBIT", 225000.0, "SUCCESS"),
    ("T004", "A004", date(2026, 1, 11), "ATM_DEBIT", 10000.0, "SUCCESS"),
    ("T005", "A005", date(2026, 1, 14), "NEFT_CREDIT", 350000.0, "SUCCESS"),
    ("T006", "A006", date(2026, 1, 18), "UPI_DEBIT", 2500.0, "FAILED"),
    ("T007", "A007", date(2026, 1, 22), "IMPS_CREDIT", 18000.0, "SUCCESS"),
    ("T008", "A008", date(2026, 1, 28), "UPI_DEBIT", 6500.0, "SUCCESS"),
    ("T009", "A009", date(2026, 2, 1), "RTGS_DEBIT", 410000.0, "SUCCESS"),
    ("T010", "A010", date(2026, 2, 4), "NEFT_CREDIT", 90000.0, "SUCCESS"),
    ("T011", "A011", date(2026, 2, 7), "NEFT_DEBIT", 85000.0, "SUCCESS"),
    ("T012", "A012", date(2026, 2, 10), "UPI_CREDIT", 12000.0, "SUCCESS"),
    ("T013", "A001", date(2026, 2, 12), "ATM_DEBIT", 15000.0, "SUCCESS"),
    ("T014", "A003", date(2026, 2, 15), "NEFT_CREDIT", 125000.0, "SUCCESS"),
    ("T015", "A005", date(2026, 2, 18), "RTGS_DEBIT", 275000.0, "SUCCESS"),
    ("T016", "A009", date(2026, 2, 21), "NEFT_CREDIT", 210000.0, "SUCCESS"),
], "transaction_id string, account_id string, transaction_date date, transaction_type string, amount double, transaction_status string")

bank_branches = make_df([
    ("B001", "Mumbai Fort", "Mumbai", "MH"), ("B002", "Bengaluru MG Road", "Bengaluru", "KA"),
    ("B003", "Ahmedabad Central", "Ahmedabad", "GJ"), ("B004", "Hyderabad Banjara", "Hyderabad", "TS"),
    ("B005", "Delhi Connaught", "Delhi", "DL"), ("B006", "Kochi Marine", "Kochi", "KL"),
    ("B007", "Kolkata Park", "Kolkata", "WB"), ("B008", "Lucknow Hazratganj", "Lucknow", "UP"),
    ("B009", "Pune Camp", "Pune", "MH"), ("B010", "Jaipur C-Scheme", "Jaipur", "RJ"),
], "branch_id string, branch_name string, city string, state string")

bank_loans = make_df([
    ("L001", "BC001", "HOME", 4200000.0, 8.45, "ACTIVE"),
    ("L002", "BC002", "VEHICLE", 850000.0, 9.10, "ACTIVE"),
    ("L003", "BC003", "BUSINESS", 7500000.0, 10.25, "ACTIVE"),
    ("L004", "BC004", "PERSONAL", 350000.0, 12.50, "CLOSED"),
    ("L005", "BC005", "BUSINESS", 12000000.0, 9.75, "ACTIVE"),
    ("L006", "BC006", "HOME", 3100000.0, 8.30, "ACTIVE"),
    ("L007", "BC008", "PERSONAL", 500000.0, 13.25, "DELINQUENT"),
    ("L008", "BC010", "VEHICLE", 1100000.0, 9.00, "ACTIVE"),
], "loan_id string, customer_id string, loan_type string, loan_amount double, interest_rate double, loan_status string")

bank_credit_cards = make_df([
    ("CC001", "BC001", "GOLD", 200000.0, 45000.0),
    ("CC002", "BC002", "CLASSIC", 100000.0, 12000.0),
    ("CC003", "BC003", "PLATINUM", 500000.0, 175000.0),
    ("CC004", "BC004", "CLASSIC", 75000.0, 8000.0),
    ("CC005", "BC005", "CORPORATE", 1000000.0, 420000.0),
    ("CC006", "BC006", "PLATINUM", 400000.0, 99000.0),
    ("CC007", "BC008", "GOLD", 180000.0, 135000.0),
    ("CC008", "BC010", "GOLD", 225000.0, 51000.0),
], "card_id string, customer_id string, card_type string, credit_limit double, outstanding_amount double")

lab_root = str(Path.cwd() / "pyspark_course_data")
print(f"Spark {spark.version} is ready.")
print("Retail and banking classroom DataFrames are loaded; Day 1 performs the first actions.")

## Project progression

```text
Retail raw data                    Banking raw data
      │                                  │
      ▼                                  ▼
   Bronze                             Bronze
      │                                  │
Quality + cleaning                 Quality + cleaning
      │                                  │
      ▼                                  ▼
   Silver                             Silver
      │                                  │
Joins + business rules            Customer + account integration
      │                                  │
Aggregates + windows              Transactions + windows
      │                                  │
      ▼                                  ▼
    Gold                               Gold
      │                                  │
Retail KPI pack                  Banking KPI assignment
```

---

# Day 1 — Introduction to PySpark

**Project milestone:** Create the retail Spark session, load customers, and verify the first five records.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain introduction to pyspark in plain language and say why a data engineer needs it.
- Recognize the core ideas: Spark, PySpark, distributed computing, driver, executors.
- Use the main APIs safely: `SparkSession.builder.getOrCreate`, `createDataFrame`, `show`, `spark.version`.
- Implement the retail requirement: create the retail Spark session, load customers, and verify the first five records.
- Apply the same idea independently to creating and inspecting a banking customer DataFrame.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: The driver creates a Spark application; executors run tasks over partitions and report status and small results back to the driver.
- Answer interview questions about PySpark, pandas, and SQL and production trade-offs.

## 2. Detailed Notes

### What is it?

PySpark is the Python interface to Apache Spark, a distributed engine that processes data by dividing work across machines.

### Why do we need it?

It lets a familiar Python program process data that is too large or too slow for one machine while keeping high-level DataFrame APIs.

### How does it work?

The Python driver builds a logical plan. Spark schedules distributed tasks, executors process partitions, and the driver coordinates the result.

### What happens inside Spark?

The driver creates a Spark application; executors run tasks over partitions and report status and small results back to the driver.

### What happens in production?

Teams use PySpark for batch ETL, lakehouse transformations, data-quality checks, feature preparation, and large analytical jobs.

### Simple analogy

Think of the driver as a restaurant manager who divides a large order among several cooks, the executors.

### Performance and design note

Keep computation distributed and avoid pulling large datasets to the driver.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `SparkSession.builder.getOrCreate` | Create or reuse a Spark session | `SparkSession.builder.appName('name').getOrCreate()` | Application entry point |
| `createDataFrame` | Build a DataFrame | `spark.createDataFrame(data, schema)` | Small samples and tests |
| `show` | Display a limited preview | `df.show(5, truncate=False)` | Interactive inspection |
| `spark.version` | Read Spark version | `spark.version` | Environment verification |

## 4. Retail Dataset — Retail customers

Primary DataFrame for today: `retail_customers`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_customers.printSchema()
retail_customers.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Create the retail Spark session, load customers, and verify the first five records.

**Plan before coding**

Create or reuse a SparkSession, construct the customer DataFrame with a known schema, preview a few rows, and count only because this teaching dataset is small.

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("RetailSalesAnalytics")
         .getOrCreate())

print("Spark version:", spark.version)
retail_customers.show(5, truncate=False)
print("Customer rows:", retail_customers.count())

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

The preview begins with customer IDs `C001`–`C005`, and the full sample contains **12 customers**.

### Business interpretation

The Spark application is ready, customer records are distributed as a DataFrame, and later retail steps can reuse the same session.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_customers` and describe each column used today.
2. [Beginner] Use Spark to answer a simple business question on `retail_customers`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine Spark with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use PySpark and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **creating and inspecting a banking customer DataFrame**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Create `bank_customers` from the supplied records and schema.
2. Display the first five customers without truncating names.
3. Print the DataFrame schema.
4. Count the customer records.
5. Select only customer ID, name, and type.
6. Explain which process is the driver in this notebook.
7. Explain what an executor would do for the count action.
8. Change the application name and verify it from `spark.sparkContext.appName`.
9. Compare a Python list with a Spark DataFrame in one paragraph.
10. Write three checks proving the sample loaded correctly.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

A bank receives 50 million customer rows daily. Describe how the driver, executors, and partitions cooperate to validate the file without collecting every record.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Using Python boolean operators | Spark Columns are expressions, not Python booleans. | Use `&`, `|`, and `~`, with each condition in parentheses. |
| Calling `collect()` on large data | All matching rows are moved to the driver. | Use `show`, `limit`, aggregations, or write the distributed result. |
| Assuming row order | Distributed DataFrames have no guaranteed order. | Use an explicit `orderBy` only when deterministic presentation is required. |
| Ignoring the schema | Inferred or drifting types can silently change results. | Declare schemas for production inputs and validate types at boundaries. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Introduction to PySpark?**  
PySpark is the Python interface to Apache Spark, a distributed engine that processes data by dividing work across machines.

**2. Why do we need it?**  
It lets a familiar Python program process data that is too large or too slow for one machine while keeping high-level DataFrame APIs.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `SparkSession.builder.getOrCreate`, `createDataFrame`, `show`, `spark.version`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Teams use PySpark for batch ETL, lakehouse transformations, data-quality checks, feature preparation, and large analytical jobs.

**5. Can you give a simple analogy?**  
Think of the driver as a restaurant manager who divides a large order among several cooks, the executors.

#### Intermediate

**1. What does Spark do internally?**  
The driver creates a Spark application; executors run tasks over partitions and report status and small results back to the driver.

**2. What is the main performance consideration?**  
Keep computation distributed and avoid pulling large datasets to the driver.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare PySpark, pandas, and SQL?**  
PySpark is distributed, pandas is normally single-machine, and SQL is a language that Spark can execute through the same engine.

**5. What common mistake would you watch for?**  
I watch for using python boolean operators. It happens because spark columns are expressions, not python booleans. I prevent it by use `&`, `|`, and `~`, with each condition in parentheses.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Create or reuse a SparkSession, construct the customer DataFrame with a known schema, preview a few rows, and count only because this teaching dataset is small.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same introduction to pyspark principle, and validate creating and inspecting a banking customer DataFrame without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Keep computation distributed and avoid pulling large datasets to the driver.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** Spark, PySpark, distributed computing, driver, executors.
- **Important functions:** `SparkSession.builder.getOrCreate`, `createDataFrame`, `show`, `spark.version`.
- **Retail problem solved:** create the retail Spark session, load customers, and verify the first five records.
- **Banking work:** you designed and validated creating and inspecting a banking customer DataFrame without receiving a copy-paste solution.
- **Interview takeaway:** PySpark is distributed, pandas is normally single-machine, and SQL is a language that Spark can execute through the same engine.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 2 — Spark Architecture

**Project milestone:** Trace how Spark filters completed orders and calculates their total value.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain spark architecture in plain language and say why a data engineer needs it.
- Recognize the core ideas: driver, executor, cluster manager, job, stage, task.
- Use the main APIs safely: `explain`, `getNumPartitions`, `count`, `sparkContext.applicationId`.
- Implement the retail requirement: trace how Spark filters completed orders and calculates their total value.
- Apply the same idea independently to identifying jobs, stages, and tasks in transaction processing.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: The driver asks a cluster manager for executors, builds a DAG, divides it at exchanges, and schedules tasks close to their input data when possible.
- Answer interview questions about jobs, stages, and tasks and production trade-offs.

## 2. Detailed Notes

### What is it?

Spark architecture is the set of processes and scheduling units that turn DataFrame code into parallel work.

### Why do we need it?

Knowing the architecture helps you read the Spark UI, explain failures, and locate bottlenecks instead of treating Spark as a black box.

### How does it work?

An action creates a job. Shuffle boundaries split the job into stages, and each stage launches one task per input partition.

### What happens inside Spark?

The driver asks a cluster manager for executors, builds a DAG, divides it at exchanges, and schedules tasks close to their input data when possible.

### What happens in production?

Engineers use the Spark UI to inspect jobs, stages, task skew, shuffle volume, executor loss, and memory pressure.

### Simple analogy

A job is a delivery order, stages are route segments separated by depots, and tasks are the individual vans serving partitions.

### Performance and design note

Large shuffles, uneven partitions, and too many tiny tasks are common architectural performance signals.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `explain` | Show logical and physical plans | `df.explain('formatted')` | Understand execution |
| `getNumPartitions` | Return partition count | `df.rdd.getNumPartitions()` | Estimate task parallelism |
| `count` | Count rows and trigger a job | `df.count()` | Demonstrate an action |
| `sparkContext.applicationId` | Identify the application | `spark.sparkContext.applicationId` | Spark UI and logs |

## 4. Retail Dataset — Retail orders

Primary DataFrame for today: `retail_orders`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_orders.printSchema()
retail_orders.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Trace how Spark filters completed orders and calculates their total value.

**Plan before coding**

Build a filter and global aggregation, inspect the physical plan before the action, then call `show()` to trigger the job.

In [ ]:
completed_orders = retail_orders.filter("order_status = 'COMPLETE'")
completed_total = completed_orders.groupBy().sum("total_amount")

print("Input partitions:", retail_orders.rdd.getNumPartitions())
completed_total.explain("formatted")
completed_total.show()

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

The physical plan contains a filter plus aggregate/exchange steps; the completed-order total is **₹160,248.00**.

### Business interpretation

Filtering can happen before the global exchange, reducing the data that must be combined for the final total.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_orders` and describe each column used today.
2. [Beginner] Use driver to answer a simple business question on `retail_orders`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine driver with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use executor and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **identifying jobs, stages, and tasks in transaction processing**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Filter successful debit transactions and call `explain('formatted')`.
2. Predict how many jobs `show()` can trigger and verify in the Spark UI if available.
3. Count the input partitions.
4. Explain where a shuffle appears in a transaction-type aggregation.
5. State how many tasks a stage normally receives from ten partitions.
6. Identify the driver process in local mode.
7. Describe the cluster manager's role in a real cluster.
8. Run `count()` and label it as a transformation or action.
9. Draw a DAG for filter → groupBy → sum.
10. Record two Spark UI metrics useful for diagnosing a slow stage.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

A transaction aggregation has one task running for 20 minutes while 199 tasks finish in seconds. Use architecture terms to explain the likely issue and the evidence you would inspect.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Calling every code line a job | Transformations are lazy and only actions create jobs. | Mark actions first, then inspect jobs and stages in the UI. |
| Equating partitions with executors | An executor can process many partitions over time. | Treat tasks as partition work and executors as worker processes. |
| Missing shuffle boundaries | Wide transformations exchange data across executors. | Look for `Exchange` nodes in the plan. |
| Assuming local mode equals a cluster | Processes and resource allocation differ. | Use local mode to learn APIs, but interpret production metrics in the real cluster. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Spark Architecture?**  
Spark architecture is the set of processes and scheduling units that turn DataFrame code into parallel work.

**2. Why do we need it?**  
Knowing the architecture helps you read the Spark UI, explain failures, and locate bottlenecks instead of treating Spark as a black box.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `explain`, `getNumPartitions`, `count`, `sparkContext.applicationId`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Engineers use the Spark UI to inspect jobs, stages, task skew, shuffle volume, executor loss, and memory pressure.

**5. Can you give a simple analogy?**  
A job is a delivery order, stages are route segments separated by depots, and tasks are the individual vans serving partitions.

#### Intermediate

**1. What does Spark do internally?**  
The driver asks a cluster manager for executors, builds a DAG, divides it at exchanges, and schedules tasks close to their input data when possible.

**2. What is the main performance consideration?**  
Large shuffles, uneven partitions, and too many tiny tasks are common architectural performance signals.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare jobs, stages, and tasks?**  
An action starts a job; shuffle boundaries divide it into stages; tasks are the parallel units, usually one per partition in a stage.

**5. What common mistake would you watch for?**  
I watch for calling every code line a job. It happens because transformations are lazy and only actions create jobs. I prevent it by mark actions first, then inspect jobs and stages in the ui.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Build a filter and global aggregation, inspect the physical plan before the action, then call `show()` to trigger the job.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same spark architecture principle, and validate identifying jobs, stages, and tasks in transaction processing without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Large shuffles, uneven partitions, and too many tiny tasks are common architectural performance signals.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** driver, executor, cluster manager, job, stage, task.
- **Important functions:** `explain`, `getNumPartitions`, `count`, `sparkContext.applicationId`.
- **Retail problem solved:** trace how Spark filters completed orders and calculates their total value.
- **Banking work:** you designed and validated identifying jobs, stages, and tasks in transaction processing without receiving a copy-paste solution.
- **Interview takeaway:** An action starts a job; shuffle boundaries divide it into stages; tasks are the parallel units, usually one per partition in a stage.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 3 — DataFrame Basics

**Project milestone:** Create product and order DataFrames with explicit schemas and inspect their structure.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain dataframe basics in plain language and say why a data engineer needs it.
- Recognize the core ideas: DataFrame, row, column, schema, data type.
- Use the main APIs safely: `createDataFrame`, `printSchema`, `show`, `columns`.
- Implement the retail requirement: create product and order DataFrames with explicit schemas and inspect their structure.
- Apply the same idea independently to creating accounts and transactions DataFrames with explicit schemas.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Catalyst analyzes column names and types, optimizes the logical plan, and produces a physical plan executed over partitions.
- Answer interview questions about DataFrames and Python collections and production trade-offs.

## 2. Detailed Notes

### What is it?

A Spark DataFrame is a distributed table with named columns and a schema that Spark can optimize.

### Why do we need it?

Schemas make transformations safer and give Spark enough type information to plan efficient execution.

### How does it work?

Spark stores a logical description of rows split into partitions; DataFrame operations build new immutable plans rather than changing data in place.

### What happens inside Spark?

Catalyst analyzes column names and types, optimizes the logical plan, and produces a physical plan executed over partitions.

### What happens in production?

Production ingestion usually declares a schema, validates required columns, and quarantines records that cannot satisfy the contract.

### Simple analogy

A DataFrame is a distributed spreadsheet whose column rules are written on a blueprint called the schema.

### Performance and design note

Explicit schemas avoid repeated inference and prevent accidental string processing for numeric or date fields.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `createDataFrame` | Create a DataFrame | `spark.createDataFrame(rows, schema)` | Tests and structured inputs |
| `printSchema` | Print the schema tree | `df.printSchema()` | Type inspection |
| `show` | Preview rows | `df.show(n, truncate=False)` | Data inspection |
| `columns` | List column names | `df.columns` | Schema validation |
| `dtypes` | List names and simple types | `df.dtypes` | Type checks |

## 4. Retail Dataset — Retail products and orders

Primary DataFrame for today: `retail_products`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_products.printSchema()
retail_products.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Create product and order DataFrames with explicit schemas and inspect their structure.

**Plan before coding**

Define a strict StructType, create rows that match it, then inspect schema and data separately.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

product_schema_demo = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), False),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
])

product_demo = spark.createDataFrame(
    [("PX1", "Demo Keyboard", "Electronics", 1999.0),
     ("PX2", "Demo Chair", "Furniture", 7499.0)],
    product_schema_demo,
)
product_demo.printSchema()
product_demo.show(truncate=False)
print("Columns:", product_demo.columns)

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

The schema shows non-null string IDs/names plus nullable category and double price; two demonstration products are displayed.

### Business interpretation

The product contract is explicit, so later arithmetic on price and joins on product ID have predictable types.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_products` and describe each column used today.
2. [Beginner] Use DataFrame to answer a simple business question on `retail_products`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine DataFrame with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use row and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **creating accounts and transactions DataFrames with explicit schemas**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Declare an accounts schema with non-null `account_id`.
2. Create the accounts DataFrame from the supplied sample.
3. Declare a transactions schema including date and amount.
4. Create the transactions DataFrame.
5. Print both schemas.
6. Show three rows from each DataFrame.
7. List each DataFrame's columns.
8. Explain the row grain of both tables.
9. Identify primary and foreign key candidates.
10. Add assertions for required columns and expected types.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

A transaction file suddenly sends `amount` as text and omits `transaction_type`. Design a schema-contract check and quarantine approach without silently changing valid records.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Row length does not match schema | The tuple has too few or too many values. | Align every row with the declared field order. |
| Wrong Python value type | A value cannot be converted to the declared Spark type. | Normalize input or choose the correct type before creation. |
| Nullable keys | The schema permits missing business identifiers. | Declare required keys non-nullable and validate them. |
| Confusing schema order | Tuple values are positional. | Prefer named `Row` objects or carefully documented schema order. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is DataFrame Basics?**  
A Spark DataFrame is a distributed table with named columns and a schema that Spark can optimize.

**2. Why do we need it?**  
Schemas make transformations safer and give Spark enough type information to plan efficient execution.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `createDataFrame`, `printSchema`, `show`, `columns`, `dtypes`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Production ingestion usually declares a schema, validates required columns, and quarantines records that cannot satisfy the contract.

**5. Can you give a simple analogy?**  
A DataFrame is a distributed spreadsheet whose column rules are written on a blueprint called the schema.

#### Intermediate

**1. What does Spark do internally?**  
Catalyst analyzes column names and types, optimizes the logical plan, and produces a physical plan executed over partitions.

**2. What is the main performance consideration?**  
Explicit schemas avoid repeated inference and prevent accidental string processing for numeric or date fields.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare DataFrames and Python collections?**  
A DataFrame is distributed, lazy, schema-aware, and optimized; Python lists are local, eager objects managed by the driver.

**5. What common mistake would you watch for?**  
I watch for row length does not match schema. It happens because the tuple has too few or too many values. I prevent it by align every row with the declared field order.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Define a strict StructType, create rows that match it, then inspect schema and data separately.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same dataframe basics principle, and validate creating accounts and transactions DataFrames with explicit schemas without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Explicit schemas avoid repeated inference and prevent accidental string processing for numeric or date fields.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** DataFrame, row, column, schema, data type.
- **Important functions:** `createDataFrame`, `printSchema`, `show`, `columns`, `dtypes`.
- **Retail problem solved:** create product and order DataFrames with explicit schemas and inspect their structure.
- **Banking work:** you designed and validated creating accounts and transactions DataFrames with explicit schemas without receiving a copy-paste solution.
- **Interview takeaway:** A DataFrame is distributed, lazy, schema-aware, and optimized; Python lists are local, eager objects managed by the driver.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 4 — Reading Data

**Project milestone:** Write classroom source files, then read customers from CSV and orders from Parquet with controlled schemas.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain reading data in plain language and say why a data engineer needs it.
- Recognize the core ideas: CSV, JSON, Parquet, schema inference, read options.
- Use the main APIs safely: `spark.read.csv`, `spark.read.json`, `spark.read.parquet`, `option`.
- Implement the retail requirement: write classroom source files, then read customers from CSV and orders from Parquet with controlled schemas.
- Apply the same idea independently to reading banking customers and accounts from governed file paths.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Text formats require parsing; Parquet supplies schema and column metadata, allowing Spark to read only required columns and sometimes skip row groups.
- Answer interview questions about CSV, JSON, and Parquet and production trade-offs.

## 2. Detailed Notes

### What is it?

Reading data converts files from a storage format into a DataFrame with columns, types, and partitions.

### Why do we need it?

Every pipeline begins at a boundary; correct schemas, options, and bad-record handling prevent downstream corruption.

### How does it work?

Spark lists input files, creates file partitions, parses records with the selected data source, and applies projection or filter pushdown where supported.

### What happens inside Spark?

Text formats require parsing; Parquet supplies schema and column metadata, allowing Spark to read only required columns and sometimes skip row groups.

### What happens in production?

Pipelines read immutable landing files, record ingestion metadata, validate counts and schema, and separate malformed records.

### Simple analogy

CSV is a labeled stack of plain paper, JSON is a flexible form, and Parquet is a cabinet organized by column.

### Performance and design note

Prefer explicit schemas and columnar formats such as Parquet for repeated analytical reads.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `spark.read.csv` | Read CSV | `spark.read.schema(s).option('header', True).csv(path)` | Delimited files |
| `spark.read.json` | Read JSON | `spark.read.schema(s).json(path)` | Nested or event data |
| `spark.read.parquet` | Read Parquet | `spark.read.parquet(path)` | Lake analytics |
| `option` | Set data-source option | `reader.option('mode', 'PERMISSIVE')` | Parser configuration |
| `schema` | Supply input schema | `reader.schema(my_schema)` | Stable production contracts |

## 4. Retail Dataset — Retail customers and orders files

Primary DataFrame for today: `retail_orders`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_orders.printSchema()
retail_orders.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Write classroom source files, then read customers from CSV and orders from Parquet with controlled schemas.

**Plan before coding**

Create reproducible local source folders, use the known customer schema for CSV, and allow Parquet to supply its stored schema.

In [ ]:
from pathlib import Path

lab_root = str(Path.cwd() / "pyspark_course_data")
retail_customers.write.mode("overwrite").option("header", True).csv(f"{lab_root}/customers_csv")
retail_orders.write.mode("overwrite").parquet(f"{lab_root}/orders_parquet")
retail_products.write.mode("overwrite").json(f"{lab_root}/products_json")

customers_from_csv = (spark.read
    .schema(retail_customers.schema)
    .option("header", True)
    .csv(f"{lab_root}/customers_csv"))
orders_from_parquet = spark.read.parquet(f"{lab_root}/orders_parquet")

customers_from_csv.show(3, truncate=False)
orders_from_parquet.printSchema()

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

Three customer rows display correctly; the Parquet schema preserves date and double types from `retail_orders`.

### Business interpretation

The pipeline has a reproducible raw input boundary and avoids expensive or unstable CSV type inference.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_orders` and describe each column used today.
2. [Beginner] Use CSV to answer a simple business question on `retail_orders`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine CSV with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use JSON and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **reading banking customers and accounts from governed file paths**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Write banking customers to a classroom CSV folder.
2. Read them back with the declared schema.
3. Write accounts to Parquet.
4. Read only account ID, type, and balance from Parquet.
5. Write three transactions as JSON and read them back.
6. Compare inferred CSV types with declared types.
7. Use a wrong delimiter once and diagnose the resulting schema.
8. Create a corrupt record and describe the chosen parse mode.
9. Validate file row counts against the in-memory samples.
10. Recommend a production format for repeated bank analytics and justify it.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

A daily bank landing zone contains late files, duplicated files, and one malformed CSV. Design an idempotent ingestion contract with file-level audit columns and quarantine behavior.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| All CSV columns become strings | No schema was supplied or inference was disabled. | Declare the production schema explicitly. |
| Header becomes a data row | The header option was omitted or wrong. | Set `header=True` and verify the first record. |
| Path not found | The driver resolves a different or misspelled storage path. | Print the configured path and validate storage permissions. |
| Many tiny input files | Each small file adds listing and task overhead. | Compact upstream or periodically optimize file layout. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Reading Data?**  
Reading data converts files from a storage format into a DataFrame with columns, types, and partitions.

**2. Why do we need it?**  
Every pipeline begins at a boundary; correct schemas, options, and bad-record handling prevent downstream corruption.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `spark.read.csv`, `spark.read.json`, `spark.read.parquet`, `option`, `schema`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Pipelines read immutable landing files, record ingestion metadata, validate counts and schema, and separate malformed records.

**5. Can you give a simple analogy?**  
CSV is a labeled stack of plain paper, JSON is a flexible form, and Parquet is a cabinet organized by column.

#### Intermediate

**1. What does Spark do internally?**  
Text formats require parsing; Parquet supplies schema and column metadata, allowing Spark to read only required columns and sometimes skip row groups.

**2. What is the main performance consideration?**  
Prefer explicit schemas and columnar formats such as Parquet for repeated analytical reads.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare CSV, JSON, and Parquet?**  
CSV is simple but weakly typed, JSON handles nested data but is verbose, and Parquet is typed, compressed, and columnar for analytics.

**5. What common mistake would you watch for?**  
I watch for all csv columns become strings. It happens because no schema was supplied or inference was disabled. I prevent it by declare the production schema explicitly.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Create reproducible local source folders, use the known customer schema for CSV, and allow Parquet to supply its stored schema.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same reading data principle, and validate reading banking customers and accounts from governed file paths without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Prefer explicit schemas and columnar formats such as Parquet for repeated analytical reads.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** CSV, JSON, Parquet, schema inference, read options.
- **Important functions:** `spark.read.csv`, `spark.read.json`, `spark.read.parquet`, `option`, `schema`.
- **Retail problem solved:** write classroom source files, then read customers from CSV and orders from Parquet with controlled schemas.
- **Banking work:** you designed and validated reading banking customers and accounts from governed file paths without receiving a copy-paste solution.
- **Interview takeaway:** CSV is simple but weakly typed, JSON handles nested data but is verbose, and Parquet is typed, compressed, and columnar for analytics.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 5 — Selecting Columns

**Project milestone:** Show a product catalogue containing product name, category, list price, and price including 18% tax.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain selecting columns in plain language and say why a data engineer needs it.
- Recognize the core ideas: select, alias, column expressions, projection.
- Use the main APIs safely: `select`, `col`, `alias`, `expr`.
- Implement the retail requirement: show a product catalogue containing product name, category, list price, and price including 18% tax.
- Apply the same idea independently to selecting account ID, account type, balance, and a derived available-view field.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Catalyst can prune unused columns so columnar readers and later operators handle less data.
- Answer interview questions about column names and Column expressions and production trade-offs.

## 2. Detailed Notes

### What is it?

Column selection chooses the fields and expressions that belong in the next DataFrame.

### Why do we need it?

Narrow schemas improve readability, reduce data movement, and create clear contracts between pipeline steps.

### How does it work?

`select` builds a projection containing existing columns or expressions; `alias` names derived outputs without changing the source DataFrame.

### What happens inside Spark?

Catalyst can prune unused columns so columnar readers and later operators handle less data.

### What happens in production?

Silver and Gold tables select, rename, cast, and order only the fields required by downstream consumers.

### Simple analogy

Selecting columns is packing only the tools needed for a job instead of carrying the whole workshop.

### Performance and design note

Project columns early, especially when reading wide Parquet or Delta tables.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `select` | Project columns or expressions | `df.select('a', F.col('b'))` | Choose output schema |
| `col` | Reference a column | `F.col('price')` | Build expressions |
| `alias` | Rename an output expression | `F.col('price').alias('unit_price')` | Business names |
| `expr` | Use a SQL expression | `F.expr('price * 0.9')` | Concise calculations |
| `selectExpr` | Select with SQL strings | `df.selectExpr('price * 0.9 AS sale_price')` | SQL-style projection |

## 4. Retail Dataset — Retail products

Primary DataFrame for today: `retail_products`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_products.printSchema()
retail_products.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Show a product catalogue containing product name, category, list price, and price including 18% tax.

**Plan before coding**

Reference only required columns, give presentation-friendly aliases, and calculate tax as a Column expression.

In [ ]:
from pyspark.sql import functions as F

product_catalogue = retail_products.select(
    F.col("product_id"),
    F.col("product_name").alias("item_name"),
    "category",
    F.col("price").alias("list_price"),
    F.round(F.col("price") * F.lit(1.18), 2).alias("price_with_tax"),
)
product_catalogue.show(12, truncate=False)

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

The result has five columns; for `Laptop Pro`, list price is ₹65,000 and price with tax is **₹76,700**.

### Business interpretation

Consumers receive a narrow catalogue with clear names and a reproducible tax calculation.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_products` and describe each column used today.
2. [Beginner] Use select to answer a simple business question on `retail_products`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine select with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use alias and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **selecting account ID, account type, balance, and a derived available-view field**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Select account ID, account type, and balance.
2. Alias `balance` as `current_balance`.
3. Add a literal currency column with value `INR`.
4. Create `balance_in_thousands` rounded to two decimals.
5. Use `selectExpr` to reproduce the calculation.
6. Return customer ID without exposing customer name.
7. Arrange output columns in a documented order.
8. Select transaction amount as `transaction_amount`.
9. Compare the schemas before and after projection.
10. Explain why early projection matters for a wide account table.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

Create a privacy-safe account extract for analytics that keeps join keys and financial measures but excludes direct customer identifiers and unnecessary operational columns.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Selecting a missing column | The name is misspelled or removed earlier. | Check `df.columns` and use analyzed errors to find the first bad step. |
| Using a Python value without `lit` | Spark expects a Column expression in many APIs. | Wrap constants with `F.lit` when needed. |
| Overwriting meaning with a poor alias | The new name hides unit or calculation semantics. | Use business names that include units or meaning. |
| Keeping every column | Wide rows increase I/O and make contracts unclear. | Select only required fields near the source. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Selecting Columns?**  
Column selection chooses the fields and expressions that belong in the next DataFrame.

**2. Why do we need it?**  
Narrow schemas improve readability, reduce data movement, and create clear contracts between pipeline steps.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `select`, `col`, `alias`, `expr`, `selectExpr`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Silver and Gold tables select, rename, cast, and order only the fields required by downstream consumers.

**5. Can you give a simple analogy?**  
Selecting columns is packing only the tools needed for a job instead of carrying the whole workshop.

#### Intermediate

**1. What does Spark do internally?**  
Catalyst can prune unused columns so columnar readers and later operators handle less data.

**2. What is the main performance consideration?**  
Project columns early, especially when reading wide Parquet or Delta tables.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare column names and Column expressions?**  
A string names an existing field; a Column expression can calculate, cast, rename, or combine values in the execution plan.

**5. What common mistake would you watch for?**  
I watch for selecting a missing column. It happens because the name is misspelled or removed earlier. I prevent it by check `df.columns` and use analyzed errors to find the first bad step.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Reference only required columns, give presentation-friendly aliases, and calculate tax as a Column expression.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same selecting columns principle, and validate selecting account ID, account type, balance, and a derived available-view field without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Project columns early, especially when reading wide Parquet or Delta tables.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** select, alias, column expressions, projection.
- **Important functions:** `select`, `col`, `alias`, `expr`, `selectExpr`.
- **Retail problem solved:** show a product catalogue containing product name, category, list price, and price including 18% tax.
- **Banking work:** you designed and validated selecting account ID, account type, balance, and a derived available-view field without receiving a copy-paste solution.
- **Interview takeaway:** A string names an existing field; a Column expression can calculate, cast, rename, or combine values in the execution plan.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 6 — Filtering Data

**Project milestone:** Find completed retail orders worth at least ₹10,000 placed from 1 February 2026 onward.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain filtering data in plain language and say why a data engineer needs it.
- Recognize the core ideas: filter, where, boolean conditions, operator precedence.
- Use the main APIs safely: `filter`, `where`, `isin`, `between`.
- Implement the retail requirement: find completed retail orders worth at least ₹10,000 placed from 1 February 2026 onward.
- Apply the same idea independently to finding successful high-value banking transactions above a configurable threshold.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Filters are added to the logical plan and may be pushed into Parquet or Delta scans, reducing rows read or processed.
- Answer interview questions about filter and where and production trade-offs.

## 2. Detailed Notes

### What is it?

Filtering keeps only rows whose condition evaluates to true.

### Why do we need it?

Most business rules target a relevant subset, such as completed high-value orders or successful debits.

### How does it work?

PySpark builds boolean Column expressions with comparisons, `&`, `|`, `~`, `isin`, and null checks; `filter` and `where` are aliases.

### What happens inside Spark?

Filters are added to the logical plan and may be pushed into Parquet or Delta scans, reducing rows read or processed.

### What happens in production?

Pipelines filter by processing date, valid status, geography, consent, and incremental-watermark ranges.

### Simple analogy

A filter is a security gate: every row must satisfy the written entry rule.

### Performance and design note

Filter early with selective predicates and inspect plans for pushdown, while preserving required reconciliation data.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `filter` | Keep matching rows | `df.filter(F.col('amount') > 1000)` | Business subsets |
| `where` | Alias for filter | `df.where('status = 'OK'')` | SQL-style conditions |
| `isin` | Match a value set | `F.col('status').isin('A', 'B')` | Status lists |
| `between` | Inclusive range | `F.col('amount').between(100, 500)` | Range rules |
| `isNull` | Test missing value | `F.col('x').isNull()` | Data quality |

## 4. Retail Dataset — Retail orders

Primary DataFrame for today: `retail_orders`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_orders.printSchema()
retail_orders.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Find completed retail orders worth at least ₹10,000 placed from 1 February 2026 onward.

**Plan before coding**

Combine three parenthesized Column conditions, then project only review fields and order the small display.

In [ ]:
high_value_orders = (retail_orders
    .filter(
        (F.col("order_status") == "COMPLETE") &
        (F.col("total_amount") >= 10000) &
        (F.col("order_date") >= F.lit("2026-02-01").cast("date"))
    )
    .select("order_id", "customer_id", "order_date", "total_amount"))

high_value_orders.orderBy("order_id").show(truncate=False)

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

Orders **O1007, O1008, O1011, and O1013** qualify; their amounts are ₹12,499, ₹18,500, ₹12,999.50, and ₹65,000.

### Business interpretation

Management can focus on recent, completed, high-value sales without mixing cancelled or pending orders.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_orders` and describe each column used today.
2. [Beginner] Use filter to answer a simple business question on `retail_orders`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine filter with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use where and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **finding successful high-value banking transactions above a configurable threshold**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Filter transactions above ₹50,000.
2. Keep only successful transactions.
3. Return only debit transactions.
4. Find credits between ₹10,000 and ₹75,000.
5. Filter two transaction types with `isin`.
6. Find transactions on or after 1 February 2026.
7. Combine date, type, status, and amount conditions.
8. Write the same rule once with `filter` and once with `where`.
9. Count rejected rows separately for reconciliation.
10. Parameterize the threshold in one variable.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

Fraud analysts need successful debit transactions above ₹75,000 made during a seven-day review window, excluding approved corporate accounts. Write the row-level filtering design and reconciliation totals.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Using `and` or `or` | Python tries to reduce a distributed Column to one boolean. | Use `&` or `|` with parenthesized expressions. |
| Missing parentheses | Python operator precedence changes the expression. | Wrap every comparison before combining conditions. |
| Comparing incompatible types | A date or amount is still a string. | Cast deliberately and inspect the schema. |
| Dropping rejected rows silently | Filtered-out data is not reconciled. | Count or quarantine rejected records as part of pipeline metrics. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Filtering Data?**  
Filtering keeps only rows whose condition evaluates to true.

**2. Why do we need it?**  
Most business rules target a relevant subset, such as completed high-value orders or successful debits.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `filter`, `where`, `isin`, `between`, `isNull`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Pipelines filter by processing date, valid status, geography, consent, and incremental-watermark ranges.

**5. Can you give a simple analogy?**  
A filter is a security gate: every row must satisfy the written entry rule.

#### Intermediate

**1. What does Spark do internally?**  
Filters are added to the logical plan and may be pushed into Parquet or Delta scans, reducing rows read or processed.

**2. What is the main performance consideration?**  
Filter early with selective predicates and inspect plans for pushdown, while preserving required reconciliation data.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare filter and where?**  
They are API aliases in PySpark; teams choose the spelling that reads best with DataFrame or SQL-style code.

**5. What common mistake would you watch for?**  
I watch for using `and` or `or`. It happens because python tries to reduce a distributed column to one boolean. I prevent it by use `&` or `|` with parenthesized expressions.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Combine three parenthesized Column conditions, then project only review fields and order the small display.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same filtering data principle, and validate finding successful high-value banking transactions above a configurable threshold without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Filter early with selective predicates and inspect plans for pushdown, while preserving required reconciliation data.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** filter, where, boolean conditions, operator precedence.
- **Important functions:** `filter`, `where`, `isin`, `between`, `isNull`.
- **Retail problem solved:** find completed retail orders worth at least ₹10,000 placed from 1 February 2026 onward.
- **Banking work:** you designed and validated finding successful high-value banking transactions above a configurable threshold without receiving a copy-paste solution.
- **Interview takeaway:** They are API aliases in PySpark; teams choose the spelling that reads best with DataFrame or SQL-style code.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 7 — Column Transformations

**Project milestone:** Calculate a 10% promotional price and classify products into budget, standard, and premium price bands.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain column transformations in plain language and say why a data engineer needs it.
- Recognize the core ideas: withColumn, cast, when, otherwise, derived columns.
- Use the main APIs safely: `withColumn`, `cast`, `when`, `otherwise`.
- Implement the retail requirement: calculate a 10% promotional price and classify products into budget, standard, and premium price bands.
- Apply the same idea independently to categorizing customers or accounts by balance with explicit numeric types.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Expressions remain inside Spark's optimized plan and are evaluated on executors, avoiding Python row-by-row loops.
- Answer interview questions about withColumn and select and production trade-offs.

## 2. Detailed Notes

### What is it?

Column transformations create or replace fields using expressions while keeping the DataFrame immutable.

### Why do we need it?

Raw values rarely match business meaning; pipelines standardize types, derive measures, and classify records.

### How does it work?

`withColumn` attaches a named expression. `cast` changes type, and chained `when` clauses implement ordered conditional logic.

### What happens inside Spark?

Expressions remain inside Spark's optimized plan and are evaluated on executors, avoiding Python row-by-row loops.

### What happens in production?

Silver layers standardize data types and statuses; Gold layers add governed measures and dimensions.

### Simple analogy

A transformation is an assembly-line station that adds a label or reshapes one part without changing the original shipment.

### Performance and design note

Prefer built-in Column expressions; collapse clear transformations and avoid repeated expensive expressions.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `withColumn` | Add or replace a column | `df.withColumn('x', expression)` | Derived fields |
| `cast` | Convert data type | `F.col('x').cast('double')` | Type standardization |
| `when` | Start conditional expression | `F.when(condition, value)` | Classification |
| `otherwise` | Set fallback value | `expr.otherwise(value)` | Complete conditions |
| `round` | Round numeric output | `F.round('amount', 2)` | Financial presentation |

## 4. Retail Dataset — Retail products

Primary DataFrame for today: `retail_products`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_products.printSchema()
retail_products.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Calculate a 10% promotional price and classify products into budget, standard, and premium price bands.

**Plan before coding**

Calculate price with a built-in expression, then evaluate price-band conditions from narrowest threshold to final fallback.

In [ ]:
enriched_products = (retail_products
    .withColumn("promo_price", F.round(F.col("price") * F.lit(0.90), 2))
    .withColumn(
        "price_band",
        F.when(F.col("price") < 1000, "BUDGET")
         .when(F.col("price") < 10000, "STANDARD")
         .otherwise("PREMIUM")
    ))

enriched_products.select(
    "product_id", "product_name", "price", "promo_price", "price_band"
).orderBy("product_id").show(12, truncate=False)

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

`P001 Laptop Pro` has promo price **₹58,500** and band **PREMIUM**; `P012 Coffee Pack` is **BUDGET**.

### Business interpretation

Merchandising receives consistent promotion values and segments that can drive catalogue campaigns.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_products` and describe each column used today.
2. [Beginner] Use withColumn to answer a simple business question on `retail_products`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine withColumn with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use cast and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **categorizing customers or accounts by balance with explicit numeric types**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Cast account balance to double in a demo copy.
2. Create `balance_band` for negative, low, medium, and high balances.
3. Create an `is_overdrawn` boolean.
4. Round balances to two decimals.
5. Add a literal currency code.
6. Replace the original balance only after comparing results.
7. Classify loan interest rates into three bands.
8. Use `otherwise` for unexpected values.
9. Count records in each balance band using only concepts learned so far where possible.
10. Document boundary behavior for exactly ₹10,000 and ₹100,000.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

Create an account health classification using balance and account type. The rules overlap; document priority so every account receives exactly one label.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Forgetting `otherwise` | Unmatched rows become null. | Provide an explicit fallback or intentionally validate null outcomes. |
| Incorrect condition order | A broad earlier rule captures rows before a specific rule. | Order rules from specific/high priority to general. |
| Accidental type loss | A cast fails and produces null values. | Compare null counts before and after casting. |
| Long withColumn chains | Repeated plan growth becomes hard to read. | Use a clear `select` for many independent derived fields. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Column Transformations?**  
Column transformations create or replace fields using expressions while keeping the DataFrame immutable.

**2. Why do we need it?**  
Raw values rarely match business meaning; pipelines standardize types, derive measures, and classify records.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `withColumn`, `cast`, `when`, `otherwise`, `round`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Silver layers standardize data types and statuses; Gold layers add governed measures and dimensions.

**5. Can you give a simple analogy?**  
A transformation is an assembly-line station that adds a label or reshapes one part without changing the original shipment.

#### Intermediate

**1. What does Spark do internally?**  
Expressions remain inside Spark's optimized plan and are evaluated on executors, avoiding Python row-by-row loops.

**2. What is the main performance consideration?**  
Prefer built-in Column expressions; collapse clear transformations and avoid repeated expensive expressions.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare withColumn and select?**  
`withColumn` is convenient for adding or replacing fields; `select` makes the entire output schema explicit and can be clearer for many columns.

**5. What common mistake would you watch for?**  
I watch for forgetting `otherwise`. It happens because unmatched rows become null. I prevent it by provide an explicit fallback or intentionally validate null outcomes.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Calculate price with a built-in expression, then evaluate price-band conditions from narrowest threshold to final fallback.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same column transformations principle, and validate categorizing customers or accounts by balance with explicit numeric types without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Prefer built-in Column expressions; collapse clear transformations and avoid repeated expensive expressions.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** withColumn, cast, when, otherwise, derived columns.
- **Important functions:** `withColumn`, `cast`, `when`, `otherwise`, `round`.
- **Retail problem solved:** calculate a 10% promotional price and classify products into budget, standard, and premium price bands.
- **Banking work:** you designed and validated categorizing customers or accounts by balance with explicit numeric types without receiving a copy-paste solution.
- **Interview takeaway:** `withColumn` is convenient for adding or replacing fields; `select` makes the entire output schema explicit and can be clearer for many columns.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 8 — String Functions

**Project milestone:** Clean padded customer names and build standardized product labels.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain string functions in plain language and say why a data engineer needs it.
- Recognize the core ideas: trim, upper, lower, concat, substring, split.
- Use the main APIs safely: `trim`, `upper`, `lower`, `concat_ws`.
- Implement the retail requirement: clean padded customer names and build standardized product labels.
- Apply the same idea independently to cleaning banking customer names and parsing structured account identifiers.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Catalyst represents string functions as JVM expressions and can optimize them together with projections and filters.
- Answer interview questions about split and substring and production trade-offs.

## 2. Detailed Notes

### What is it?

String functions clean, standardize, parse, and combine text columns using Spark-native expressions.

### Why do we need it?

Whitespace, casing, embedded codes, and inconsistent identifiers cause failed joins and misleading groups.

### How does it work?

Built-in string functions operate on each row inside executors and return new Columns without moving data to Python.

### What happens inside Spark?

Catalyst represents string functions as JVM expressions and can optimize them together with projections and filters.

### What happens in production?

Teams normalize keys, parse source codes, create display labels, and validate identifier patterns in Silver data.

### Simple analogy

String cleaning is preparing mailing labels: trim edges, standardize capitalization, and separate address parts.

### Performance and design note

Use built-in functions instead of Python UDFs and avoid regex when simpler functions are sufficient.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `trim` | Remove surrounding spaces | `F.trim('name')` | Clean text |
| `upper` | Convert to uppercase | `F.upper('code')` | Canonical codes |
| `lower` | Convert to lowercase | `F.lower('email')` | Case normalization |
| `concat_ws` | Join values with separator | `F.concat_ws(' - ', 'id', 'name')` | Labels |
| `substring` | Extract character range | `F.substring('id', 1, 3)` | Fixed-format codes |
| `split` | Split text to array | `F.split('value', '-')` | Delimited identifiers |

## 4. Retail Dataset — Retail customers and products

Primary DataFrame for today: `retail_customers`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_customers.printSchema()
retail_customers.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Clean padded customer names and build standardized product labels.

**Plan before coding**

Create a controlled dirty input, trim before uppercasing, and combine normalized state with an ID fragment.

In [ ]:
dirty_customers = retail_customers.withColumn(
    "customer_name", F.concat(F.lit("  "), F.col("customer_name"), F.lit(" "))
)

clean_customers = dirty_customers.select(
    "customer_id",
    F.upper(F.trim("customer_name")).alias("customer_name_clean"),
    F.concat_ws("-", F.upper("state"), F.substring("customer_id", 2, 3)).alias("customer_label"),
)
clean_customers.show(5, truncate=False)

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

`C001 / Aarav Sharma / KA` becomes name **AARAV SHARMA** and label **KA-001**.

### Business interpretation

Consistent text improves matching, grouping, exports, and downstream customer search.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_customers` and describe each column used today.
2. [Beginner] Use trim to answer a simple business question on `retail_customers`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine trim with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use upper and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **cleaning banking customer names and parsing structured account identifiers**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Trim banking customer names.
2. Create an uppercase customer type.
3. Create a lowercase display version of city.
4. Build `customer_id - customer_name` labels.
5. Extract the numeric part of customer IDs.
6. Split a demo branch code such as `KA-BLR-001`.
7. Standardize account types to uppercase.
8. Find names whose cleaned length is under three characters.
9. Compare distinct counts before and after normalization.
10. Define rules that prevent accidental changes to case-sensitive identifiers.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

Two bank systems represent names and branch codes differently. Create a canonical matching key while retaining original fields and explain collision risks.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Joining dirty strings | Spaces or case differences prevent equality. | Create and validate canonical keys before the join. |
| Wrong substring position | Spark substring positions start at 1. | Test boundary cases with short sample strings. |
| Regex overuse | Complex expressions are slower and harder to maintain. | Use trim, split, replace, or substring when possible. |
| Destroying original text | Cleaning overwrites audit evidence. | Retain raw and cleaned columns through the quality layer. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is String Functions?**  
String functions clean, standardize, parse, and combine text columns using Spark-native expressions.

**2. Why do we need it?**  
Whitespace, casing, embedded codes, and inconsistent identifiers cause failed joins and misleading groups.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `trim`, `upper`, `lower`, `concat_ws`, `substring`, `split`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Teams normalize keys, parse source codes, create display labels, and validate identifier patterns in Silver data.

**5. Can you give a simple analogy?**  
String cleaning is preparing mailing labels: trim edges, standardize capitalization, and separate address parts.

#### Intermediate

**1. What does Spark do internally?**  
Catalyst represents string functions as JVM expressions and can optimize them together with projections and filters.

**2. What is the main performance consideration?**  
Use built-in functions instead of Python UDFs and avoid regex when simpler functions are sufficient.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare split and substring?**  
`split` uses a delimiter or regex to make an array; `substring` extracts characters from fixed positions.

**5. What common mistake would you watch for?**  
I watch for joining dirty strings. It happens because spaces or case differences prevent equality. I prevent it by create and validate canonical keys before the join.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Create a controlled dirty input, trim before uppercasing, and combine normalized state with an ID fragment.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same string functions principle, and validate cleaning banking customer names and parsing structured account identifiers without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Use built-in functions instead of Python UDFs and avoid regex when simpler functions are sufficient.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** trim, upper, lower, concat, substring, split.
- **Important functions:** `trim`, `upper`, `lower`, `concat_ws`, `substring`, `split`.
- **Retail problem solved:** clean padded customer names and build standardized product labels.
- **Banking work:** you designed and validated cleaning banking customer names and parsing structured account identifiers without receiving a copy-paste solution.
- **Interview takeaway:** `split` uses a delimiter or regex to make an array; `substring` extracts characters from fixed positions.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 9 — Date Functions

**Project milestone:** Summarize orders by year and month and calculate days from order to a fixed classroom review date.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain date functions in plain language and say why a data engineer needs it.
- Recognize the core ideas: date types, current_date, datediff, date_add, months_between, year, month.
- Use the main APIs safely: `to_date`, `current_date`, `datediff`, `date_add`.
- Implement the retail requirement: summarize orders by year and month and calculate days from order to a fixed classroom review date.
- Apply the same idea independently to analyzing banking transactions by month and computing account/customer tenure.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Typed date values are stored compactly and evaluated by Spark expressions; partition filters can be pushed down when types align.
- Answer interview questions about date and timestamp and production trade-offs.

## 2. Detailed Notes

### What is it?

Date functions represent calendar values with real date types and derive periods, ages, and relative dates.

### Why do we need it?

Business reporting depends on correct day and month boundaries; strings cannot safely provide calendar arithmetic.

### How does it work?

Spark parses strings with an explicit pattern and applies calendar-aware expressions such as difference, addition, and extraction.

### What happens inside Spark?

Typed date values are stored compactly and evaluated by Spark expressions; partition filters can be pushed down when types align.

### What happens in production?

Incremental loads, retention rules, month-end reports, and service-level checks all depend on governed timestamps and time zones.

### Simple analogy

A date type is a real calendar; a date string is only a label that may or may not follow the calendar's rules.

### Performance and design note

Filter typed partition columns with typed literals and avoid wrapping them unnecessarily in functions during pruning.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `to_date` | Parse or cast to date | `F.to_date('text', 'yyyy-MM-dd')` | Input standardization |
| `current_date` | Current session date | `F.current_date()` | Age and SLA logic |
| `datediff` | Difference in days | `F.datediff('end', 'start')` | Elapsed days |
| `date_add` | Add calendar days | `F.date_add('date', 7)` | Due dates |
| `months_between` | Fractional months | `F.months_between('end', 'start')` | Tenure |
| `year / month` | Extract period | `F.year('date')` | Reporting dimensions |

## 4. Retail Dataset — Retail orders

Primary DataFrame for today: `retail_orders`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_orders.printSchema()
retail_orders.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Summarize orders by year and month and calculate days from order to a fixed classroom review date.

**Plan before coding**

Use a fixed review date for reproducible output, extract reporting periods, and calculate elapsed and follow-up days.

In [ ]:
review_date = F.lit("2026-03-01").cast("date")

dated_orders = retail_orders.select(
    "order_id", "order_date", "total_amount",
    F.year("order_date").alias("order_year"),
    F.month("order_date").alias("order_month"),
    F.datediff(review_date, F.col("order_date")).alias("days_to_review"),
    F.date_add("order_date", 7).alias("follow_up_date"),
)
dated_orders.orderBy("order_date").show(15, truncate=False)

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

Order `O1001` on 2026-01-03 has review age **57 days** and follow-up date **2026-01-10**.

### Business interpretation

Retail teams can create stable monthly partitions, age orders, and schedule follow-up activities.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_orders` and describe each column used today.
2. [Beginner] Use date types to answer a simple business question on `retail_orders`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine date types with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use current_date and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **analyzing banking transactions by month and computing account/customer tenure**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Extract transaction year and month.
2. Create a `yyyy-MM` reporting label.
3. Calculate days from transaction to 1 March 2026.
4. Add seven days to create a review deadline.
5. Calculate customer tenure in months.
6. Filter February 2026 transactions.
7. Identify transactions older than 30 days at the fixed review date.
8. Explain date versus timestamp for ATM events.
9. Create a month-start column.
10. Validate that every transaction date is on or after account opening in a joined production design.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

Design a timezone-safe daily transaction report for events produced in multiple countries, including the business date used for branch reporting.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Date remains a string | Parsing or casting was skipped. | Use `to_date` with an explicit expected pattern. |
| Wrong parsing pattern | `dd-MM` and `MM-dd` are confused. | Document source formats and quarantine invalid parses. |
| Using current date in tests | Expected output changes each day. | Inject a fixed as-of date for reproducible tests. |
| Ignoring time zones | Timestamp boundaries move across regions. | Set session timezone and define the business timezone explicitly. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Date Functions?**  
Date functions represent calendar values with real date types and derive periods, ages, and relative dates.

**2. Why do we need it?**  
Business reporting depends on correct day and month boundaries; strings cannot safely provide calendar arithmetic.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `to_date`, `current_date`, `datediff`, `date_add`, `months_between`, `year / month`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Incremental loads, retention rules, month-end reports, and service-level checks all depend on governed timestamps and time zones.

**5. Can you give a simple analogy?**  
A date type is a real calendar; a date string is only a label that may or may not follow the calendar's rules.

#### Intermediate

**1. What does Spark do internally?**  
Typed date values are stored compactly and evaluated by Spark expressions; partition filters can be pushed down when types align.

**2. What is the main performance consideration?**  
Filter typed partition columns with typed literals and avoid wrapping them unnecessarily in functions during pruning.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare date and timestamp?**  
A date stores a calendar day; a timestamp stores an instant or local date-time and requires explicit timezone handling.

**5. What common mistake would you watch for?**  
I watch for date remains a string. It happens because parsing or casting was skipped. I prevent it by use `to_date` with an explicit expected pattern.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Use a fixed review date for reproducible output, extract reporting periods, and calculate elapsed and follow-up days.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same date functions principle, and validate analyzing banking transactions by month and computing account/customer tenure without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Filter typed partition columns with typed literals and avoid wrapping them unnecessarily in functions during pruning.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** date types, current_date, datediff, date_add, months_between, year, month.
- **Important functions:** `to_date`, `current_date`, `datediff`, `date_add`, `months_between`, `year / month`.
- **Retail problem solved:** summarize orders by year and month and calculate days from order to a fixed classroom review date.
- **Banking work:** you designed and validated analyzing banking transactions by month and computing account/customer tenure without receiving a copy-paste solution.
- **Interview takeaway:** A date stores a calendar day; a timestamp stores an instant or local date-time and requires explicit timezone handling.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 10 — Aggregations

**Project milestone:** Calculate completed-order count, revenue, average order value, minimum, and maximum order values.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain aggregations in plain language and say why a data engineer needs it.
- Recognize the core ideas: count, sum, average, minimum, maximum, global aggregation.
- Use the main APIs safely: `count`, `sum`, `avg`, `min`.
- Implement the retail requirement: calculate completed-order count, revenue, average order value, minimum, and maximum order values.
- Apply the same idea independently to calculating transaction totals and averages by the intended status population.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Spark commonly uses partial and final aggregate operators so each executor reduces data before network transfer.
- Answer interview questions about count, countDistinct, and approximate distinct and production trade-offs.

## 2. Detailed Notes

### What is it?

An aggregation reduces many input rows to summary measures such as count, total, average, minimum, or maximum.

### Why do we need it?

KPIs and reconciliation controls summarize detailed events into information that people and systems can compare.

### How does it work?

Aggregate functions collect partial results inside partitions, shuffle or merge them, and produce final values.

### What happens inside Spark?

Spark commonly uses partial and final aggregate operators so each executor reduces data before network transfer.

### What happens in production?

Pipelines calculate file counts, financial totals, data-quality metrics, and dashboard measures at declared grains.

### Simple analogy

Instead of carrying every receipt to head office, each store totals its receipts and sends one subtotal for final addition.

### Performance and design note

Use mergeable built-in aggregates, avoid unnecessary distinct counts, and keep the result grain explicit.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `count` | Count rows/non-null values | `F.count('*')` | Volume KPI |
| `sum` | Add numeric values | `F.sum('amount')` | Revenue |
| `avg` | Arithmetic mean | `F.avg('amount')` | Average order value |
| `min` | Smallest value | `F.min('amount')` | Range check |
| `max` | Largest value | `F.max('amount')` | Peak transaction |
| `agg` | Calculate several aggregates | `df.agg(F.sum('x'), F.avg('x'))` | One summary row |

## 4. Retail Dataset — Retail orders

Primary DataFrame for today: `retail_orders`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_orders.printSchema()
retail_orders.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Calculate completed-order count, revenue, average order value, minimum, and maximum order values.

**Plan before coding**

Filter the correct business population first, then compute five named measures in one aggregation.

In [ ]:
order_kpis = (retail_orders
    .filter(F.col("order_status") == "COMPLETE")
    .agg(
        F.count("*").alias("completed_orders"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("total_amount"), 2).alias("average_order_value"),
        F.min("total_amount").alias("minimum_order_value"),
        F.max("total_amount").alias("maximum_order_value"),
    ))
order_kpis.show(truncate=False)

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

There are **13** completed orders, total revenue **₹160,248.00**, average **₹12,326.77**, minimum **₹1,000**, and maximum **₹65,000**.

### Business interpretation

The KPI row reconciles completed business only; cancelled and pending demand remain outside recognized revenue.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_orders` and describe each column used today.
2. [Beginner] Use count to answer a simple business question on `retail_orders`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine count with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use sum and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **calculating transaction totals and averages by the intended status population**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Count all transaction rows.
2. Count successful transaction rows.
3. Sum successful transaction amount.
4. Calculate average successful debit amount.
5. Find minimum and maximum credit amounts.
6. Count distinct accounts with activity.
7. Explain null behavior in `count(column)`.
8. Create one KPI row with five measures.
9. Reconcile successful plus failed counts to total count.
10. Document whether amounts are signed or separated by transaction type.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

Create auditable daily control totals for a payment file: record count, amount total, unique accounts, rejected count, and min/max event time, with a stated status population.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Aggregating the wrong population | Status or date filters were omitted. | Define the KPI population before writing expressions. |
| Using floating point blindly | Binary doubles can introduce financial rounding artifacts. | Use appropriate decimal types for production money. |
| Misreading count(column) | Null values in that column are skipped. | Use `count('*')` for row count and name both metrics clearly. |
| Losing the business grain | A global total is compared with grouped totals. | Document one row per what for every result. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Aggregations?**  
An aggregation reduces many input rows to summary measures such as count, total, average, minimum, or maximum.

**2. Why do we need it?**  
KPIs and reconciliation controls summarize detailed events into information that people and systems can compare.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `count`, `sum`, `avg`, `min`, `max`, `agg`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Pipelines calculate file counts, financial totals, data-quality metrics, and dashboard measures at declared grains.

**5. Can you give a simple analogy?**  
Instead of carrying every receipt to head office, each store totals its receipts and sends one subtotal for final addition.

#### Intermediate

**1. What does Spark do internally?**  
Spark commonly uses partial and final aggregate operators so each executor reduces data before network transfer.

**2. What is the main performance consideration?**  
Use mergeable built-in aggregates, avoid unnecessary distinct counts, and keep the result grain explicit.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare count, countDistinct, and approximate distinct?**  
`count` counts rows or non-null values, `countDistinct` gives exact unique counts with shuffle cost, and approximate methods trade small error for scale.

**5. What common mistake would you watch for?**  
I watch for aggregating the wrong population. It happens because status or date filters were omitted. I prevent it by define the kpi population before writing expressions.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Filter the correct business population first, then compute five named measures in one aggregation.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same aggregations principle, and validate calculating transaction totals and averages by the intended status population without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Use mergeable built-in aggregates, avoid unnecessary distinct counts, and keep the result grain explicit.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** count, sum, average, minimum, maximum, global aggregation.
- **Important functions:** `count`, `sum`, `avg`, `min`, `max`, `agg`.
- **Retail problem solved:** calculate completed-order count, revenue, average order value, minimum, and maximum order values.
- **Banking work:** you designed and validated calculating transaction totals and averages by the intended status population without receiving a copy-paste solution.
- **Interview takeaway:** `count` counts rows or non-null values, `countDistinct` gives exact unique counts with shuffle cost, and approximate methods trade small error for scale.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 11 — GroupBy and Multiple Aggregations

**Project milestone:** Calculate completed-order revenue, order count, and average order value for each store.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain groupby and multiple aggregations in plain language and say why a data engineer needs it.
- Recognize the core ideas: groupBy, grouping key, multiple aggregates, result grain.
- Use the main APIs safely: `groupBy`, `agg`, `countDistinct`, `orderBy`.
- Implement the retail requirement: calculate completed-order revenue, order count, and average order value for each store.
- Apply the same idea independently to summarizing transaction count and amount by branch and transaction type.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: A hash or sort aggregate usually surrounds an exchange partitioned by the grouping columns.
- Answer interview questions about global and grouped aggregation and production trade-offs.

## 2. Detailed Notes

### What is it?

Grouped aggregation creates one summary row for each unique combination of grouping keys.

### Why do we need it?

Businesses need KPIs by store, category, branch, month, customer segment, and other dimensions.

### How does it work?

Rows with the same key are brought together logically; Spark calculates partial summaries and shuffles by key for final aggregation.

### What happens inside Spark?

A hash or sort aggregate usually surrounds an exchange partitioned by the grouping columns.

### What happens in production?

Gold tables define stable dimensions and measures, then test uniqueness at the declared grain.

### Simple analogy

Receipts are sorted into labeled baskets by store and category before each basket is counted and totaled.

### Performance and design note

Choose low-to-moderate-cardinality grouping keys, handle skew, and avoid grouping by unnecessary high-cardinality text.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `groupBy` | Define grouping keys | `df.groupBy('store_id')` | Dimensional KPIs |
| `agg` | Apply named measures | `grouped.agg(F.sum('amount'))` | Multiple KPIs |
| `countDistinct` | Exact unique count | `F.countDistinct('order_id')` | Unique orders |
| `orderBy` | Sort final presentation | `df.orderBy(F.desc('revenue'))` | Ranked report display |

## 4. Retail Dataset — Retail orders by store

Primary DataFrame for today: `retail_orders`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_orders.printSchema()
retail_orders.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Calculate completed-order revenue, order count, and average order value for each store.

**Plan before coding**

Filter recognized sales, group at one row per store, calculate three measures, and sort only for presentation.

In [ ]:
store_sales = (retail_orders
    .filter(F.col("order_status") == "COMPLETE")
    .groupBy("store_id")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.round(F.sum("total_amount"), 2).alias("revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_order_value"),
    )
    .orderBy(F.desc("revenue")))
store_sales.show(truncate=False)

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

Store `S001` leads with revenue **₹100,648.50** from 5 completed orders; `S002` follows with **₹41,499.50**.

### Business interpretation

Store managers can compare both volume and value; revenue alone would hide different order patterns.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_orders` and describe each column used today.
2. [Beginner] Use groupBy to answer a simple business question on `retail_orders`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine groupBy with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use grouping key and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **summarizing transaction count and amount by branch and transaction type**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Join-free warm-up: group transactions by transaction type.
2. Calculate count, sum, and average amount by type.
3. Group accounts by branch and account type.
4. Calculate total and average balance by branch.
5. Use two grouping columns.
6. Order groups by total amount descending.
7. Add a distinct-account measure.
8. Reconcile grouped totals to the global total.
9. State the row grain in a markdown cell.
10. Identify a grouping key likely to cause high cardinality.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

Create a branch control report with account counts and balances, then describe how transaction facts would later be integrated without double-counting account balances.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Selecting ungrouped detail columns | They are neither grouping keys nor aggregates. | Add them to the group only if they belong to the intended grain. |
| Grouping by too many columns | The result approaches detail-level cardinality. | Define the business grain first. |
| Double-counting after a later join | A one-to-many relationship repeats measures. | Aggregate each fact at compatible grain before joining. |
| Sorting huge intermediate data | Global order requires expensive movement. | Sort only final bounded results when required. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is GroupBy and Multiple Aggregations?**  
Grouped aggregation creates one summary row for each unique combination of grouping keys.

**2. Why do we need it?**  
Businesses need KPIs by store, category, branch, month, customer segment, and other dimensions.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `groupBy`, `agg`, `countDistinct`, `orderBy`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Gold tables define stable dimensions and measures, then test uniqueness at the declared grain.

**5. Can you give a simple analogy?**  
Receipts are sorted into labeled baskets by store and category before each basket is counted and totaled.

#### Intermediate

**1. What does Spark do internally?**  
A hash or sort aggregate usually surrounds an exchange partitioned by the grouping columns.

**2. What is the main performance consideration?**  
Choose low-to-moderate-cardinality grouping keys, handle skew, and avoid grouping by unnecessary high-cardinality text.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare global and grouped aggregation?**  
A global aggregation returns one summary row; grouped aggregation returns one row per distinct key combination.

**5. What common mistake would you watch for?**  
I watch for selecting ungrouped detail columns. It happens because they are neither grouping keys nor aggregates. I prevent it by add them to the group only if they belong to the intended grain.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Filter recognized sales, group at one row per store, calculate three measures, and sort only for presentation.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same groupby and multiple aggregations principle, and validate summarizing transaction count and amount by branch and transaction type without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Choose low-to-moderate-cardinality grouping keys, handle skew, and avoid grouping by unnecessary high-cardinality text.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** groupBy, grouping key, multiple aggregates, result grain.
- **Important functions:** `groupBy`, `agg`, `countDistinct`, `orderBy`.
- **Retail problem solved:** calculate completed-order revenue, order count, and average order value for each store.
- **Banking work:** you designed and validated summarizing transaction count and amount by branch and transaction type without receiving a copy-paste solution.
- **Interview takeaway:** A global aggregation returns one summary row; grouped aggregation returns one row per distinct key combination.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 12 — Joins

**Project milestone:** Produce completed-order details with customer and store names, while preserving and reporting unmatched keys.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain joins in plain language and say why a data engineer needs it.
- Recognize the core ideas: inner join, left join, right join, full join, semi join, anti join.
- Use the main APIs safely: `join`, `alias`, `left_semi`, `left_anti`.
- Implement the retail requirement: produce completed-order details with customer and store names, while preserving and reporting unmatched keys.
- Apply the same idea independently to joining customers to accounts and integrating branches without duplicating account balances.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Large joins usually exchange both sides by key; small-side broadcast can avoid that shuffle. Catalyst selects a strategy from statistics and hints.
- Answer interview questions about inner, left, semi, and anti joins and production trade-offs.

## 2. Detailed Notes

### What is it?

A join combines rows from two DataFrames by matching key expressions.

### Why do we need it?

Business facts and dimensions live in separate tables; joins integrate customers, orders, products, stores, and accounts.

### How does it work?

Spark matches keys using a physical strategy such as broadcast hash, shuffled hash, or sort-merge join, while join type controls unmatched rows.

### What happens inside Spark?

Large joins usually exchange both sides by key; small-side broadcast can avoid that shuffle. Catalyst selects a strategy from statistics and hints.

### What happens in production?

Engineers validate key uniqueness and relationship cardinality before joining, then reconcile row counts and unmatched keys.

### Simple analogy

A join is matching claim tickets: the join type decides whether unmatched people, unmatched items, or both remain in the report.

### Performance and design note

Project and filter early, broadcast genuinely small dimensions, and measure key skew and match rates.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `join` | Combine DataFrames | `left.join(right, 'key', 'left')` | Fact-dimension integration |
| `alias` | Qualify DataFrames | `df.alias('o')` | Resolve ambiguous columns |
| `left_semi` | Keep left matches | `df.join(dim, key, 'left_semi')` | Existence filter |
| `left_anti` | Keep left non-matches | `df.join(dim, key, 'left_anti')` | Data-quality exceptions |

## 4. Retail Dataset — Retail customers, orders, and stores

Primary DataFrame for today: `retail_orders`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_orders.printSchema()
retail_orders.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Produce completed-order details with customer and store names, while preserving and reporting unmatched keys.

**Plan before coding**

Filter the fact first, left-join two dimensions on qualified keys, select a narrow schema, and audit unmatched customers with an anti join.

In [ ]:
order_details = (retail_orders.alias("o")
    .filter(F.col("o.order_status") == "COMPLETE")
    .join(retail_customers.alias("c"), F.col("o.customer_id") == F.col("c.customer_id"), "left")
    .join(retail_stores.alias("s"), F.col("o.store_id") == F.col("s.store_id"), "left")
    .select(
        F.col("o.order_id"), F.col("o.order_date"),
        F.col("c.customer_name"), F.col("s.store_name"),
        F.col("o.total_amount"),
    ))

order_details.orderBy("order_id").show(20, truncate=False)

unmatched_customers = retail_orders.join(retail_customers, "customer_id", "left_anti")
print("Orders with unknown customer:", unmatched_customers.count())

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

All 13 completed orders receive names; the unmatched-customer audit count is **0**.

### Business interpretation

The integrated result is human-readable, while the anti-join control protects against silently losing invalid foreign keys.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_orders` and describe each column used today.
2. [Beginner] Use inner join to answer a simple business question on `retail_orders`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine inner join with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use left join and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **joining customers to accounts and integrating branches without duplicating account balances**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Inner-join customers with accounts.
2. Left-join customers with accounts and find customers without accounts.
3. Join accounts to branches.
4. Select qualified columns after a three-table join.
5. Count accounts before and after dimension joins.
6. Use a left-semi join to find customers with loans.
7. Use a left-anti join to find customers without loans.
8. Explain why joining transactions changes account row counts.
9. Check customer-key uniqueness before joining.
10. Produce a match-rate metric for branch IDs.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

Build a customer-account-branch view and a separate unmatched-key exception table. State relationship cardinalities and prove balances are not double-counted.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Ambiguous column error | Both sides contain the same non-key name. | Alias DataFrames and select qualified columns. |
| Unexpected row multiplication | The key is duplicated on one or both sides. | Profile key uniqueness and declare join cardinality. |
| Lost unmatched records | An inner join was used without business approval. | Choose join type from requirements and audit anti-join rows. |
| Null join keys do not match | Standard equality does not match nulls. | Validate or intentionally use null-safe equality where appropriate. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Joins?**  
A join combines rows from two DataFrames by matching key expressions.

**2. Why do we need it?**  
Business facts and dimensions live in separate tables; joins integrate customers, orders, products, stores, and accounts.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `join`, `alias`, `left_semi`, `left_anti`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Engineers validate key uniqueness and relationship cardinality before joining, then reconcile row counts and unmatched keys.

**5. Can you give a simple analogy?**  
A join is matching claim tickets: the join type decides whether unmatched people, unmatched items, or both remain in the report.

#### Intermediate

**1. What does Spark do internally?**  
Large joins usually exchange both sides by key; small-side broadcast can avoid that shuffle. Catalyst selects a strategy from statistics and hints.

**2. What is the main performance consideration?**  
Project and filter early, broadcast genuinely small dimensions, and measure key skew and match rates.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare inner, left, semi, and anti joins?**  
Inner keeps matches, left keeps every left row, semi keeps left rows that have a match without right columns, and anti keeps left rows with no match.

**5. What common mistake would you watch for?**  
I watch for ambiguous column error. It happens because both sides contain the same non-key name. I prevent it by alias dataframes and select qualified columns.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Filter the fact first, left-join two dimensions on qualified keys, select a narrow schema, and audit unmatched customers with an anti join.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same joins principle, and validate joining customers to accounts and integrating branches without duplicating account balances without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Project and filter early, broadcast genuinely small dimensions, and measure key skew and match rates.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** inner join, left join, right join, full join, semi join, anti join.
- **Important functions:** `join`, `alias`, `left_semi`, `left_anti`.
- **Retail problem solved:** produce completed-order details with customer and store names, while preserving and reporting unmatched keys.
- **Banking work:** you designed and validated joining customers to accounts and integrating branches without duplicating account balances without receiving a copy-paste solution.
- **Interview takeaway:** Inner keeps matches, left keeps every left row, semi keeps left rows that have a match without right columns, and anti keeps left rows with no match.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 13 — Union and Data Combination

**Project milestone:** Combine online and offline order feeds safely and remove an intentionally duplicated order by business key.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain union and data combination in plain language and say why a data engineer needs it.
- Recognize the core ideas: union, unionByName, schema alignment, distinct, deduplication.
- Use the main APIs safely: `union`, `unionByName`, `distinct`, `dropDuplicates`.
- Implement the retail requirement: combine online and offline order feeds safely and remove an intentionally duplicated order by business key.
- Apply the same idea independently to combining NEFT and UPI transaction feeds with lineage and business-key deduplication.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Union is usually narrow because partitions are concatenated; a following `distinct` requires a wide shuffle to find duplicates.
- Answer interview questions about union and unionByName and production trade-offs.

## 2. Detailed Notes

### What is it?

A union stacks rows from compatible DataFrames into one result.

### Why do we need it?

Pipelines combine channels, regions, daily batches, or source systems before common processing.

### How does it work?

`union` aligns columns by position; `unionByName` aligns by name and can fill missing columns when explicitly allowed.

### What happens inside Spark?

Union is usually narrow because partitions are concatenated; a following `distinct` requires a wide shuffle to find duplicates.

### What happens in production?

Bronze tables append source batches with lineage columns, and controlled deduplication uses documented business keys.

### Simple analogy

Union is stacking two decks of cards; they must use the same fields, and duplicate removal means comparing the entire stack.

### Performance and design note

Prefer `unionByName`, align types deliberately, and avoid full-row distinct when a business-key strategy is available.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `union` | Stack by position | `df1.union(df2)` | Identical schemas |
| `unionByName` | Stack by column name | `df1.unionByName(df2)` | Safer batch combination |
| `distinct` | Remove duplicate rows | `df.distinct()` | Exact full-row deduplication |
| `dropDuplicates` | Deduplicate by keys | `df.dropDuplicates(['order_id'])` | Business-key cleanup |

## 4. Retail Dataset — Online and store retail orders

Primary DataFrame for today: `retail_orders`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_orders.printSchema()
retail_orders.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Combine online and offline order feeds safely and remove an intentionally duplicated order by business key.

**Plan before coding**

Add a lineage column, intentionally vary column order, combine by name, inject a duplicate, then deduplicate by order ID.

In [ ]:
online_orders = (retail_orders
    .filter(F.col("store_id") == "S004")
    .withColumn("sales_channel", F.lit("ONLINE")))
offline_orders = (retail_orders
    .filter(F.col("store_id") != "S004")
    .withColumn("sales_channel", F.lit("STORE"))
    .select("sales_channel", *retail_orders.columns))  # deliberately different order

combined_orders = (online_orders
    .unionByName(offline_orders)
    .unionByName(online_orders.filter(F.col("order_id") == "O1003"))
    .dropDuplicates(["order_id"]))

print("Combined unique orders:", combined_orders.count())
combined_orders.groupBy("sales_channel").count().show()

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

The final result has **15 unique orders**: 3 online and 12 store orders.

### Business interpretation

Channel feeds can evolve in column order without corrupting values, and replayed order IDs do not duplicate the table.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_orders` and describe each column used today.
2. [Beginner] Use union to answer a simple business question on `retail_orders`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine union with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use unionByName and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **combining NEFT and UPI transaction feeds with lineage and business-key deduplication**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Split supplied transactions into two type-based feeds.
2. Add a `source_system` column to both feeds.
3. Reorder one feed's columns.
4. Combine feeds with `unionByName`.
5. Explain why positional `union` is risky.
6. Add one replayed transaction record.
7. Remove duplicates by transaction ID.
8. Compare row counts before and after deduplication.
9. Create a schema-difference check before union.
10. State how you would handle a new optional source column.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

Three payment feeds arrive with evolving schemas and occasional replayed files. Design schema alignment, lineage, and deterministic deduplication rules.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Values appear under wrong columns | Positional union used different column order. | Use `unionByName` and validate schemas. |
| Union type mismatch | Same-name columns have incompatible types. | Cast to a canonical contract before combining. |
| Distinct is unexpectedly expensive | It shuffles and compares full rows. | Deduplicate on documented keys with deterministic rules. |
| Legitimate repeats removed | The chosen key was too broad. | Define event identity with business owners. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Union and Data Combination?**  
A union stacks rows from compatible DataFrames into one result.

**2. Why do we need it?**  
Pipelines combine channels, regions, daily batches, or source systems before common processing.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `union`, `unionByName`, `distinct`, `dropDuplicates`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Bronze tables append source batches with lineage columns, and controlled deduplication uses documented business keys.

**5. Can you give a simple analogy?**  
Union is stacking two decks of cards; they must use the same fields, and duplicate removal means comparing the entire stack.

#### Intermediate

**1. What does Spark do internally?**  
Union is usually narrow because partitions are concatenated; a following `distinct` requires a wide shuffle to find duplicates.

**2. What is the main performance consideration?**  
Prefer `unionByName`, align types deliberately, and avoid full-row distinct when a business-key strategy is available.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare union and unionByName?**  
`union` matches by position and can silently misplace values; `unionByName` matches names and is safer for evolving schemas.

**5. What common mistake would you watch for?**  
I watch for values appear under wrong columns. It happens because positional union used different column order. I prevent it by use `unionbyname` and validate schemas.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Add a lineage column, intentionally vary column order, combine by name, inject a duplicate, then deduplicate by order ID.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same union and data combination principle, and validate combining NEFT and UPI transaction feeds with lineage and business-key deduplication without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Prefer `unionByName`, align types deliberately, and avoid full-row distinct when a business-key strategy is available.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** union, unionByName, schema alignment, distinct, deduplication.
- **Important functions:** `union`, `unionByName`, `distinct`, `dropDuplicates`.
- **Retail problem solved:** combine online and offline order feeds safely and remove an intentionally duplicated order by business key.
- **Banking work:** you designed and validated combining NEFT and UPI transaction feeds with lineage and business-key deduplication without receiving a copy-paste solution.
- **Interview takeaway:** `union` matches by position and can silently misplace values; `unionByName` matches names and is safer for evolving schemas.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 14 — Null Handling

**Project milestone:** Standardize missing category values, reject rows without product IDs, and preserve an auditable quality flag.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain null handling in plain language and say why a data engineer needs it.
- Recognize the core ideas: null, isNull, fillna, dropna, replace, three-valued logic.
- Use the main APIs safely: `isNull / isNotNull`, `fillna`, `dropna`, `replace`.
- Implement the retail requirement: standardize missing category values, reject rows without product IDs, and preserve an auditable quality flag.
- Apply the same idea independently to handling missing account and transaction values under financial data-quality policies.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Nullability is part of the schema; Spark expressions propagate nulls unless a function or condition defines another outcome.
- Answer interview questions about null, blank, and zero and production trade-offs.

## 2. Detailed Notes

### What is it?

Null represents an unknown or missing value, not zero, blank text, or false.

### Why do we need it?

Missing values affect comparisons, aggregates, joins, and business decisions, so each column needs an explicit policy.

### How does it work?

Spark uses SQL three-valued logic. Functions can detect, fill, drop, or replace values, but the correct action depends on business meaning.

### What happens inside Spark?

Nullability is part of the schema; Spark expressions propagate nulls unless a function or condition defines another outcome.

### What happens in production?

Data-quality rules classify missing fields as reject, default, impute, preserve, or escalate, with metrics by source and date.

### Simple analogy

Null is an unanswered question, while zero is a clear answer; replacing every unanswered question with zero changes the story.

### Performance and design note

Apply targeted column policies and avoid repeated full scans solely for ad hoc null counts.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `isNull / isNotNull` | Test null state | `F.col('x').isNull()` | Quality rules |
| `fillna` | Fill null values | `df.fillna({'city': 'UNKNOWN'})` | Approved defaults |
| `dropna` | Drop rows with nulls | `df.dropna(subset=['id'])` | Reject invalid keys |
| `replace` | Replace known values | `df.replace('N/A', None, ['city'])` | Sentinel cleanup |
| `coalesce` | First non-null expression | `F.coalesce('a', 'b')` | Fallback logic |

## 4. Retail Dataset — Retail products with controlled missing values

Primary DataFrame for today: `retail_products`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_products.printSchema()
retail_products.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Standardize missing category values, reject rows without product IDs, and preserve an auditable quality flag.

**Plan before coding**

Convert a sentinel to real null, flag quality before filling, apply approved text defaults, and reject missing keys.

In [ ]:
quality_input = spark.createDataFrame([
    ("P900", "Sample Cable", None, 399.0),
    (None, "Unknown Item", "N/A", 100.0),
    ("P901", None, "Accessories", None),
], ["product_id", "product_name", "category", "price"])

standardized = (quality_input
    .replace("N/A", None, subset=["category"])
    .withColumn("had_missing_value",
                F.col("product_name").isNull() | F.col("category").isNull() | F.col("price").isNull())
    .fillna({"product_name": "UNKNOWN PRODUCT", "category": "UNCATEGORIZED"})
    .dropna(subset=["product_id"]))

standardized.show(truncate=False)

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

Two rows remain (`P900`, `P901`); both keep `had_missing_value=true`, and the row with missing product ID is rejected.

### Business interpretation

The curated table is usable without pretending that missing source values were originally known.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_products` and describe each column used today.
2. [Beginner] Use null to answer a simple business question on `retail_products`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine null with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use isNull and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **handling missing account and transaction values under financial data-quality policies**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Count nulls in each account column.
2. Replace blank account types with null.
3. Reject rows missing account ID.
4. Do not replace missing financial amounts with zero; explain why.
5. Fill missing branch ID only if an approved `UNKNOWN` member exists.
6. Create a missing-value flag before filling.
7. Use `coalesce` for preferred and fallback city.
8. Compare `count('*')` with `count('amount')`.
9. Create accepted and rejected DataFrames.
10. Report null rates as data-quality metrics.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

Design null policies for account ID, branch ID, transaction amount, customer city, and loan interest rate, including which records are rejected versus preserved.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Using `== None` | SQL null comparison does not behave like Python equality. | Use `isNull` or `isNotNull`. |
| Filling every numeric null with zero | Unknown money becomes a real amount. | Use a field-specific business policy. |
| Dropping rows without metrics | Data loss becomes invisible. | Split accepted/rejected data and report counts. |
| Losing the original quality state | Filling removes evidence of missingness. | Create flags or retain raw fields before remediation. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Null Handling?**  
Null represents an unknown or missing value, not zero, blank text, or false.

**2. Why do we need it?**  
Missing values affect comparisons, aggregates, joins, and business decisions, so each column needs an explicit policy.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `isNull / isNotNull`, `fillna`, `dropna`, `replace`, `coalesce`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Data-quality rules classify missing fields as reject, default, impute, preserve, or escalate, with metrics by source and date.

**5. Can you give a simple analogy?**  
Null is an unanswered question, while zero is a clear answer; replacing every unanswered question with zero changes the story.

#### Intermediate

**1. What does Spark do internally?**  
Nullability is part of the schema; Spark expressions propagate nulls unless a function or condition defines another outcome.

**2. What is the main performance consideration?**  
Apply targeted column policies and avoid repeated full scans solely for ad hoc null counts.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare null, blank, and zero?**  
Null means unknown, blank is a known empty string, and zero is a numeric value; they require different business treatment.

**5. What common mistake would you watch for?**  
I watch for using `== none`. It happens because sql null comparison does not behave like python equality. I prevent it by use `isnull` or `isnotnull`.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Convert a sentinel to real null, flag quality before filling, apply approved text defaults, and reject missing keys.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same null handling principle, and validate handling missing account and transaction values under financial data-quality policies without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Apply targeted column policies and avoid repeated full scans solely for ad hoc null counts.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** null, isNull, fillna, dropna, replace, three-valued logic.
- **Important functions:** `isNull / isNotNull`, `fillna`, `dropna`, `replace`, `coalesce`.
- **Retail problem solved:** standardize missing category values, reject rows without product IDs, and preserve an auditable quality flag.
- **Banking work:** you designed and validated handling missing account and transaction values under financial data-quality policies without receiving a copy-paste solution.
- **Interview takeaway:** Null means unknown, blank is a known empty string, and zero is a numeric value; they require different business treatment.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 15 — Window Functions

**Project milestone:** Rank products by completed sales value within each category and keep the top two.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain window functions in plain language and say why a data engineer needs it.
- Recognize the core ideas: Window specification, partitionBy, orderBy, row_number, rank, dense_rank.
- Use the main APIs safely: `Window.partitionBy`, `orderBy`, `row_number`, `rank`.
- Implement the retail requirement: rank products by completed sales value within each category and keep the top two.
- Apply the same idea independently to finding the highest-value transactions per customer after joining accounts.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Spark exchanges rows by window partition and sorts within each partition before evaluating window expressions.
- Answer interview questions about row_number, rank, and dense_rank and production trade-offs.

## 2. Detailed Notes

### What is it?

A window function calculates across related rows while keeping each row in the result.

### Why do we need it?

Ranking, top-N, running context, and within-group comparisons cannot be expressed by ordinary groupBy without losing detail.

### How does it work?

A Window specification defines the group and order; ranking functions assign positions according to tie behavior.

### What happens inside Spark?

Spark exchanges rows by window partition and sorts within each partition before evaluating window expressions.

### What happens in production?

Gold analytics use windows for top products, latest records, customer sequences, deduplication, and percentile analysis.

### Simple analogy

A race awards places within each age group while keeping every runner's row visible.

### Performance and design note

Partition on meaningful keys, limit window width, avoid single global partitions, and reduce rows before expensive sorts.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `Window.partitionBy` | Define independent groups | `Window.partitionBy('category')` | Per-group analytics |
| `orderBy` | Define row sequence | `window.orderBy(F.desc('sales'))` | Ranking order |
| `row_number` | Unique sequential number | `F.row_number().over(w)` | Latest-row selection |
| `rank` | Rank with gaps | `F.rank().over(w)` | Competition ranking |
| `dense_rank` | Rank without gaps | `F.dense_rank().over(w)` | Top distinct values |

## 4. Retail Dataset — Retail order items joined to products

Primary DataFrame for today: `retail_order_items`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_order_items.printSchema()
retail_order_items.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Rank products by completed sales value within each category and keep the top two.

**Plan before coding**

Join facts to dimensions, calculate line sales, aggregate to product-category grain, then rank only that compact result within category.

In [ ]:
from pyspark.sql.window import Window

product_sales = (retail_order_items.alias("i")
    .join(retail_orders.select("order_id", "order_status"), "order_id", "inner")
    .filter(F.col("order_status") == "COMPLETE")
    .join(retail_products.select("product_id", "product_name", "category"), "product_id", "inner")
    .withColumn("line_sales", F.col("quantity") * F.col("unit_price") * (1 - F.col("discount")))
    .groupBy("category", "product_id", "product_name")
    .agg(F.round(F.sum("line_sales"), 2).alias("sales")))

category_window = Window.partitionBy("category").orderBy(F.desc("sales"), F.asc("product_id"))
top_products = (product_sales
    .withColumn("category_rank", F.dense_rank().over(category_window))
    .filter(F.col("category_rank") <= 2))

top_products.orderBy("category", "category_rank", "product_id").show(30, truncate=False)

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

Each category returns at most two product ranks; Electronics is led by `Laptop Pro`, followed by `Smartphone X`.

### Business interpretation

Category managers can compare leaders fairly within their own product groups instead of against the entire catalogue.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_order_items` and describe each column used today.
2. [Beginner] Use Window specification to answer a simple business question on `retail_order_items`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine Window specification with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use partitionBy and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **finding the highest-value transactions per customer after joining accounts**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Join transactions to accounts to obtain customer ID.
2. Keep successful transactions.
3. Define a customer-partitioned descending amount window.
4. Add `row_number`.
5. Add `rank` and `dense_rank` for comparison.
6. Return each customer's top transaction.
7. Return top three distinct amounts per customer.
8. Add transaction ID as a deterministic tie-breaker for row number.
9. Explain the output grain.
10. Inspect the physical plan for exchange and sort.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

Fraud analysts want each customer's three largest successful debits per calendar month, retaining ties and deterministic evidence rows. Design the grain and ranking rule.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| No partitionBy | All rows enter one global window. | Partition by the business group such as customer or category. |
| Non-deterministic row_number | The order columns do not break ties. | Add a stable unique tie-breaker. |
| Ranking detail before aggregation | Duplicate lines distort product ranking. | Aggregate to the intended ranking grain first. |
| Confusing rank semantics | Ties and gaps do not match the requirement. | Choose row_number, rank, or dense_rank explicitly. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Window Functions?**  
A window function calculates across related rows while keeping each row in the result.

**2. Why do we need it?**  
Ranking, top-N, running context, and within-group comparisons cannot be expressed by ordinary groupBy without losing detail.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `Window.partitionBy`, `orderBy`, `row_number`, `rank`, `dense_rank`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Gold analytics use windows for top products, latest records, customer sequences, deduplication, and percentile analysis.

**5. Can you give a simple analogy?**  
A race awards places within each age group while keeping every runner's row visible.

#### Intermediate

**1. What does Spark do internally?**  
Spark exchanges rows by window partition and sorts within each partition before evaluating window expressions.

**2. What is the main performance consideration?**  
Partition on meaningful keys, limit window width, avoid single global partitions, and reduce rows before expensive sorts.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare row_number, rank, and dense_rank?**  
`row_number` is always unique, `rank` leaves gaps after ties, and `dense_rank` gives tied rows the same rank without gaps.

**5. What common mistake would you watch for?**  
I watch for no partitionby. It happens because all rows enter one global window. I prevent it by partition by the business group such as customer or category.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Join facts to dimensions, calculate line sales, aggregate to product-category grain, then rank only that compact result within category.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same window functions principle, and validate finding the highest-value transactions per customer after joining accounts without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Partition on meaningful keys, limit window width, avoid single global partitions, and reduce rows before expensive sorts.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** Window specification, partitionBy, orderBy, row_number, rank, dense_rank.
- **Important functions:** `Window.partitionBy`, `orderBy`, `row_number`, `rank`, `dense_rank`.
- **Retail problem solved:** rank products by completed sales value within each category and keep the top two.
- **Banking work:** you designed and validated finding the highest-value transactions per customer after joining accounts without receiving a copy-paste solution.
- **Interview takeaway:** `row_number` is always unique, `rank` leaves gaps after ties, and `dense_rank` gives tied rows the same rank without gaps.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 16 — Advanced Window Functions

**Project milestone:** Compare each completed order with the customer's previous order and calculate cumulative spend.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain advanced window functions in plain language and say why a data engineer needs it.
- Recognize the core ideas: lag, lead, cumulative sum, window frame, sequence analysis.
- Use the main APIs safely: `lag`, `lead`, `rowsBetween`, `sum over window`.
- Implement the retail requirement: compare each completed order with the customer's previous order and calculate cumulative spend.
- Apply the same idea independently to comparing consecutive transactions and calculating cumulative account activity.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Spark performs the same exchange-and-sort foundation as ranking, then evaluates frame state while scanning each ordered partition.
- Answer interview questions about lag, lead, and cumulative aggregates and production trade-offs.

## 2. Detailed Notes

### What is it?

Advanced windows access neighboring rows or cumulative frames within an ordered group.

### Why do we need it?

Change detection, running totals, time gaps, and next-event analysis require context from a sequence.

### How does it work?

`lag` and `lead` read relative rows; aggregate functions with `rowsBetween` calculate over a defined frame.

### What happens inside Spark?

Spark performs the same exchange-and-sort foundation as ranking, then evaluates frame state while scanning each ordered partition.

### What happens in production?

Teams calculate balance movements, customer purchase gaps, session behavior, cumulative exposure, and slowly changing record changes.

### Simple analogy

A runner looks at the athlete immediately ahead and behind while also checking the cumulative distance completed so far.

### Performance and design note

Reuse compatible window specifications and ensure large customer or account groups do not create extreme skew.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `lag` | Read previous row value | `F.lag('amount').over(w)` | Change analysis |
| `lead` | Read next row value | `F.lead('date').over(w)` | Next-event timing |
| `rowsBetween` | Define row frame | `w.rowsBetween(Window.unboundedPreceding, Window.currentRow)` | Running totals |
| `sum over window` | Cumulative total | `F.sum('amount').over(frame)` | Running measures |

## 4. Retail Dataset — Retail customer order sequence

Primary DataFrame for today: `retail_orders`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_orders.printSchema()
retail_orders.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Compare each completed order with the customer's previous order and calculate cumulative spend.

**Plan before coding**

Create one deterministic customer sequence, derive a running frame from it, and reuse both for neighbor and cumulative metrics.

In [ ]:
customer_sequence = Window.partitionBy("customer_id").orderBy("order_date", "order_id")
running_frame = customer_sequence.rowsBetween(Window.unboundedPreceding, Window.currentRow)

order_sequence = (retail_orders
    .filter(F.col("order_status") == "COMPLETE")
    .select("customer_id", "order_id", "order_date", "total_amount")
    .withColumn("previous_amount", F.lag("total_amount").over(customer_sequence))
    .withColumn("next_order_date", F.lead("order_date").over(customer_sequence))
    .withColumn("change_from_previous", F.round(F.col("total_amount") - F.col("previous_amount"), 2))
    .withColumn("cumulative_spend", F.round(F.sum("total_amount").over(running_frame), 2)))

order_sequence.orderBy("customer_id", "order_date", "order_id").show(30, truncate=False)

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

A customer's first order has null previous amount; customer `C001` progresses from ₹2,499 to cumulative spend **₹67,499** after order `O1013`.

### Business interpretation

Marketing can detect spend acceleration and time to next purchase without collapsing individual orders.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_orders` and describe each column used today.
2. [Beginner] Use lag to answer a simple business question on `retail_orders`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine lag with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use lead and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **comparing consecutive transactions and calculating cumulative account activity**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Partition transactions by account and order by date plus ID.
2. Add previous transaction amount.
3. Add next transaction date.
4. Calculate change from previous amount.
5. Calculate days to next transaction.
6. Create cumulative debit amount.
7. Create cumulative credit amount.
8. Keep first-event nulls rather than forcing zero without explanation.
9. Check tie behavior for same-day events.
10. Explain how this differs from groupBy.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

Detect accounts with three rapidly increasing debits in sequence. Define deterministic event order, comparison logic, and evidence columns.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Default frame misunderstood | Ordered aggregate frames may not match row-based intent. | Declare `rowsBetween` explicitly. |
| Tied timestamps | Sequence is non-deterministic. | Add a stable event ID to ordering. |
| Wrong partition key | Values leak across customers or accounts. | State the entity whose history is independent. |
| Replacing first lag with zero blindly | No previous event is not the same as zero. | Preserve null or add a first-event flag. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Advanced Window Functions?**  
Advanced windows access neighboring rows or cumulative frames within an ordered group.

**2. Why do we need it?**  
Change detection, running totals, time gaps, and next-event analysis require context from a sequence.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `lag`, `lead`, `rowsBetween`, `sum over window`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Teams calculate balance movements, customer purchase gaps, session behavior, cumulative exposure, and slowly changing record changes.

**5. Can you give a simple analogy?**  
A runner looks at the athlete immediately ahead and behind while also checking the cumulative distance completed so far.

#### Intermediate

**1. What does Spark do internally?**  
Spark performs the same exchange-and-sort foundation as ranking, then evaluates frame state while scanning each ordered partition.

**2. What is the main performance consideration?**  
Reuse compatible window specifications and ensure large customer or account groups do not create extreme skew.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare lag, lead, and cumulative aggregates?**  
`lag` reads a previous row, `lead` reads a following row, and a framed aggregate summarizes a range such as all rows to the current row.

**5. What common mistake would you watch for?**  
I watch for default frame misunderstood. It happens because ordered aggregate frames may not match row-based intent. I prevent it by declare `rowsbetween` explicitly.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Create one deterministic customer sequence, derive a running frame from it, and reuse both for neighbor and cumulative metrics.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same advanced window functions principle, and validate comparing consecutive transactions and calculating cumulative account activity without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Reuse compatible window specifications and ensure large customer or account groups do not create extreme skew.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** lag, lead, cumulative sum, window frame, sequence analysis.
- **Important functions:** `lag`, `lead`, `rowsBetween`, `sum over window`.
- **Retail problem solved:** compare each completed order with the customer's previous order and calculate cumulative spend.
- **Banking work:** you designed and validated comparing consecutive transactions and calculating cumulative account activity without receiving a copy-paste solution.
- **Interview takeaway:** `lag` reads a previous row, `lead` reads a following row, and a framed aggregate summarizes a range such as all rows to the current row.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 17 — User-Defined Functions (UDFs)

**Project milestone:** Demonstrate customer segmentation with a tested Python function, then show the preferred built-in equivalent.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain user-defined functions (udfs) in plain language and say why a data engineer needs it.
- Recognize the core ideas: Python UDF, return type, serialization, built-in alternatives.
- Use the main APIs safely: `udf`, `@F.udf`, `pandas_udf`, `when`.
- Implement the retail requirement: demonstrate customer segmentation with a tested Python function, then show the preferred built-in equivalent.
- Apply the same idea independently to creating a documented banking risk classification only when built-ins are insufficient.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Regular Python UDFs create a serialization boundary and are less visible to Catalyst than native expressions.
- Answer interview questions about built-in expressions and Python UDFs and production trade-offs.

## 2. Detailed Notes

### What is it?

A Python UDF wraps custom Python logic so it can be applied as a Spark column expression.

### Why do we need it?

UDFs fill gaps when a rule cannot be expressed reasonably with built-in Spark functions.

### How does it work?

Spark sends column values between the JVM execution engine and Python workers, runs the function, and converts results to the declared return type.

### What happens inside Spark?

Regular Python UDFs create a serialization boundary and are less visible to Catalyst than native expressions.

### What happens in production?

Teams govern UDF ownership, input/output contracts, null behavior, tests, and performance; built-ins remain the first choice.

### Simple analogy

A UDF is sending a package to a specialist workshop outside the main factory line, then bringing the result back.

### Performance and design note

Prefer built-ins, SQL expressions, or higher-order functions; benchmark unavoidable UDFs and consider vectorized alternatives when appropriate.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `udf` | Register Python function as Column function | `F.udf(fn, StringType())` | Unavoidable custom logic |
| `@F.udf` | Decorator syntax | `@F.udf('string')` | Reusable UDF declaration |
| `pandas_udf` | Vectorized Arrow UDF | `@F.pandas_udf('double')` | Vectorizable custom logic |
| `when` | Native conditional alternative | `F.when(condition, value)` | Preferred classification |

## 4. Retail Dataset — Retail customer spending

Primary DataFrame for today: `retail_customers`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_customers.printSchema()
retail_customers.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Demonstrate customer segmentation with a tested Python function, then show the preferred built-in equivalent.

**Plan before coding**

Write a pure function with null behavior, declare the return type, apply it once, and compare with an equivalent native expression.

In [ ]:
from pyspark.sql.types import StringType

def segment_customer(total_spend):
    if total_spend is None:
        return "UNKNOWN"
    if total_spend >= 30000:
        return "PLATINUM"
    if total_spend >= 10000:
        return "GOLD"
    return "STANDARD"

segment_udf = F.udf(segment_customer, StringType())

customer_spend = (retail_orders
    .filter(F.col("order_status") == "COMPLETE")
    .groupBy("customer_id")
    .agg(F.sum("total_amount").alias("total_spend")))

segmented = customer_spend.withColumn("udf_segment", segment_udf("total_spend"))
preferred = segmented.withColumn(
    "builtin_segment",
    F.when(F.col("total_spend") >= 30000, "PLATINUM")
     .when(F.col("total_spend") >= 10000, "GOLD")
     .otherwise("STANDARD"))
preferred.orderBy(F.desc("total_spend")).show(truncate=False)

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

Customer `C001` has total spend **₹67,499** and receives **PLATINUM** from both implementations.

### Business interpretation

The business rule works, but the built-in version is preferred because Spark can optimize it directly.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_customers` and describe each column used today.
2. [Beginner] Use Python UDF to answer a simple business question on `retail_customers`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine Python UDF with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use return type and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **creating a documented banking risk classification only when built-ins are insufficient**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Write a pure risk function with null handling.
2. Declare its Spark return type.
3. Apply it to loan amount and interest rate.
4. Create test cases for every rule boundary.
5. Compare output with a built-in `when` implementation.
6. Explain serialization overhead.
7. Inspect the plan for a Python evaluation node.
8. Avoid reading external mutable state inside the UDF.
9. Record unexpected inputs as `UNKNOWN`.
10. Recommend whether the rule should remain a UDF.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

A bank has a complex, versioned risk formula maintained by model governance. Design a safe UDF interface, test strategy, version field, and performance validation.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| No return type | Spark cannot reliably encode the Python result. | Declare the exact Spark return type. |
| UDF fails on null | Python logic assumes every input is present. | Define and test null behavior explicitly. |
| Using UDF for simple conditions | Native optimization is lost unnecessarily. | Use built-in expressions whenever possible. |
| Capturing large objects | The closure is serialized to workers. | Keep UDF dependencies small and deterministic. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is User-Defined Functions (UDFs)?**  
A Python UDF wraps custom Python logic so it can be applied as a Spark column expression.

**2. Why do we need it?**  
UDFs fill gaps when a rule cannot be expressed reasonably with built-in Spark functions.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `udf`, `@F.udf`, `pandas_udf`, `when`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Teams govern UDF ownership, input/output contracts, null behavior, tests, and performance; built-ins remain the first choice.

**5. Can you give a simple analogy?**  
A UDF is sending a package to a specialist workshop outside the main factory line, then bringing the result back.

#### Intermediate

**1. What does Spark do internally?**  
Regular Python UDFs create a serialization boundary and are less visible to Catalyst than native expressions.

**2. What is the main performance consideration?**  
Prefer built-ins, SQL expressions, or higher-order functions; benchmark unavoidable UDFs and consider vectorized alternatives when appropriate.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare built-in expressions and Python UDFs?**  
Built-ins run inside Spark's optimized engine; Python UDFs cross a language boundary and hide logic from many optimizations.

**5. What common mistake would you watch for?**  
I watch for no return type. It happens because spark cannot reliably encode the python result. I prevent it by declare the exact spark return type.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Write a pure function with null behavior, declare the return type, apply it once, and compare with an equivalent native expression.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same user-defined functions (udfs) principle, and validate creating a documented banking risk classification only when built-ins are insufficient without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Prefer built-ins, SQL expressions, or higher-order functions; benchmark unavoidable UDFs and consider vectorized alternatives when appropriate.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** Python UDF, return type, serialization, built-in alternatives.
- **Important functions:** `udf`, `@F.udf`, `pandas_udf`, `when`.
- **Retail problem solved:** demonstrate customer segmentation with a tested Python function, then show the preferred built-in equivalent.
- **Banking work:** you designed and validated creating a documented banking risk classification only when built-ins are insufficient without receiving a copy-paste solution.
- **Interview takeaway:** Built-ins run inside Spark's optimized engine; Python UDFs cross a language boundary and hide logic from many optimizations.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 18 — RDD Fundamentals

**Project milestone:** Parse raw pipe-delimited retail order text and calculate completed sales by store with an RDD.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain rdd fundamentals in plain language and say why a data engineer needs it.
- Recognize the core ideas: RDD, transformation, action, pair RDD, DataFrame versus RDD.
- Use the main APIs safely: `parallelize`, `map`, `filter`, `reduceByKey`.
- Implement the retail requirement: parse raw pipe-delimited retail order text and calculate completed sales by store with an RDD.
- Apply the same idea independently to processing raw banking transaction records with a small RDD demonstration.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Spark schedules RDD lineage as stages and tasks, recomputing lost partitions from lineage when possible.
- Answer interview questions about RDDs and DataFrames and production trade-offs.

## 2. Detailed Notes

### What is it?

An RDD is Spark's lower-level distributed collection of objects, divided into partitions and transformed lazily.

### Why do we need it?

RDD knowledge explains Spark's foundations and helps with genuinely unstructured records or specialized algorithms, though DataFrames are preferred for structured ETL.

### How does it work?

Transformations such as `map` and `filter` create lineage; actions such as `collect` or `count` trigger partition computation.

### What happens inside Spark?

Spark schedules RDD lineage as stages and tasks, recomputing lost partitions from lineage when possible.

### What happens in production?

Most data engineering stays with DataFrames; RDDs appear in legacy code, low-level parsing, and custom partition operations.

### Simple analogy

An RDD is a distributed pile of index cards; you describe how each worker should transform its pile.

### Performance and design note

Avoid Python object overhead for structured data and never collect an unbounded RDD to the driver.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `parallelize` | Create an RDD | `sc.parallelize(data, 2)` | Small demonstrations |
| `map` | Transform every element | `rdd.map(function)` | Record parsing |
| `filter` | Keep elements | `rdd.filter(predicate)` | Record selection |
| `reduceByKey` | Aggregate pair values | `pairs.reduceByKey(lambda a,b: a+b)` | Keyed totals |
| `collect` | Return all elements to driver | `rdd.collect()` | Only bounded teaching results |

## 4. Retail Dataset — Raw retail text records

Primary DataFrame for today: `retail_orders`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_orders.printSchema()
retail_orders.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Parse raw pipe-delimited retail order text and calculate completed sales by store with an RDD.

**Plan before coding**

Parallelize a bounded sample, parse fields, filter status, form `(store, amount)` pairs, and reduce locally before collecting two totals.

In [ ]:
raw_order_lines = [
    "O2001|S001|COMPLETE|1200.0",
    "O2002|S002|CANCELLED|800.0",
    "O2003|S001|COMPLETE|2300.0",
    "O2004|S002|COMPLETE|1500.0",
]

raw_rdd = spark.sparkContext.parallelize(raw_order_lines, 2)
store_totals_rdd = (raw_rdd
    .map(lambda line: line.split("|"))
    .filter(lambda parts: parts[2] == "COMPLETE")
    .map(lambda parts: (parts[1], float(parts[3])))
    .reduceByKey(lambda left, right: left + right))

print(sorted(store_totals_rdd.collect()))  # safe only because this result has two rows

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

The result is `[('S001', 3500.0), ('S002', 1500.0)]`.

### Business interpretation

The exercise exposes low-level distributed mechanics, while a production structured pipeline should normally use DataFrames.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_orders` and describe each column used today.
2. [Beginner] Use RDD to answer a simple business question on `retail_orders`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine RDD with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use transformation and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **processing raw banking transaction records with a small RDD demonstration**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Create an RDD from five pipe-delimited transactions.
2. Split each record into fields.
3. Filter successful records.
4. Convert amount text to float safely.
5. Create account-amount pairs.
6. Reduce amounts by account.
7. Count input records.
8. Identify malformed records in a separate RDD.
9. Convert valid parsed rows to a DataFrame.
10. Explain why the DataFrame is preferable after parsing.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

A legacy mainframe feed contains mixed record types and malformed lines. Design an RDD parsing boundary that returns typed DataFrames for valid records and rejects with reason codes.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Collecting raw data | Every object moves to driver memory. | Aggregate or write distributed results; collect only bounded outputs. |
| Parsing without validation | Malformed lines cause task failures. | Check field count and types, and route rejects separately. |
| Using groupByKey for sums | All values move across the network before reduction. | Use `reduceByKey` or DataFrame aggregation. |
| Keeping structured work in RDDs | Catalyst and compact encodings are lost. | Convert to a typed DataFrame after low-level parsing. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is RDD Fundamentals?**  
An RDD is Spark's lower-level distributed collection of objects, divided into partitions and transformed lazily.

**2. Why do we need it?**  
RDD knowledge explains Spark's foundations and helps with genuinely unstructured records or specialized algorithms, though DataFrames are preferred for structured ETL.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `parallelize`, `map`, `filter`, `reduceByKey`, `collect`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Most data engineering stays with DataFrames; RDDs appear in legacy code, low-level parsing, and custom partition operations.

**5. Can you give a simple analogy?**  
An RDD is a distributed pile of index cards; you describe how each worker should transform its pile.

#### Intermediate

**1. What does Spark do internally?**  
Spark schedules RDD lineage as stages and tasks, recomputing lost partitions from lineage when possible.

**2. What is the main performance consideration?**  
Avoid Python object overhead for structured data and never collect an unbounded RDD to the driver.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare RDDs and DataFrames?**  
RDDs offer low-level object control but little schema optimization; DataFrames are schema-aware, Catalyst-optimized, and usually faster for ETL.

**5. What common mistake would you watch for?**  
I watch for collecting raw data. It happens because every object moves to driver memory. I prevent it by aggregate or write distributed results; collect only bounded outputs.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Parallelize a bounded sample, parse fields, filter status, form `(store, amount)` pairs, and reduce locally before collecting two totals.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same rdd fundamentals principle, and validate processing raw banking transaction records with a small RDD demonstration without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Avoid Python object overhead for structured data and never collect an unbounded RDD to the driver.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** RDD, transformation, action, pair RDD, DataFrame versus RDD.
- **Important functions:** `parallelize`, `map`, `filter`, `reduceByKey`, `collect`.
- **Retail problem solved:** parse raw pipe-delimited retail order text and calculate completed sales by store with an RDD.
- **Banking work:** you designed and validated processing raw banking transaction records with a small RDD demonstration without receiving a copy-paste solution.
- **Interview takeaway:** RDDs offer low-level object control but little schema optimization; DataFrames are schema-aware, Catalyst-optimized, and usually faster for ETL.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 19 — Spark Execution and Lazy Evaluation

**Project milestone:** Build a lazy completed-sales pipeline, inspect its plan, and identify the action and shuffle boundary.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain spark execution and lazy evaluation in plain language and say why a data engineer needs it.
- Recognize the core ideas: lazy evaluation, logical plan, DAG, narrow transformation, wide transformation, action.
- Use the main APIs safely: `explain`, `count`, `take`, `write`.
- Implement the retail requirement: build a lazy completed-sales pipeline, inspect its plan, and identify the action and shuffle boundary.
- Apply the same idea independently to tracing a lazy banking transaction pipeline from transformation to action.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Catalyst rewrites DataFrame plans, and exchanges mark shuffle boundaries that split stages in the DAG scheduler.
- Answer interview questions about narrow and wide transformations and production trade-offs.

## 2. Detailed Notes

### What is it?

Lazy evaluation means Spark records transformations as a plan and computes them only when an action needs a result.

### Why do we need it?

Spark can optimize the full chain, avoid unused work, and pipeline compatible operations.

### How does it work?

Transformations extend lineage; an action triggers analysis, optimization, physical planning, stage creation, and task execution.

### What happens inside Spark?

Catalyst rewrites DataFrame plans, and exchanges mark shuffle boundaries that split stages in the DAG scheduler.

### What happens in production?

Engineers use `explain`, Spark UI, event logs, and metrics to connect source code to actual execution.

### Simple analogy

You write a complete shopping list before visiting the store, allowing one optimized trip instead of traveling after every item.

### Performance and design note

Inspect physical plans, minimize repeated actions, and persist only reused expensive plans.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `explain` | Print execution plans | `df.explain('formatted')` | Plan inspection |
| `count` | Trigger computation | `df.count()` | Action demonstration |
| `take` | Return bounded rows | `df.take(5)` | Small action |
| `write` | Materialize output | `df.write.parquet(path)` | Pipeline action |

## 4. Retail Dataset — Retail transformation pipeline

Primary DataFrame for today: `retail_orders`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_orders.printSchema()
retail_orders.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Build a lazy completed-sales pipeline, inspect its plan, and identify the action and shuffle boundary.

**Plan before coding**

Define a chain without intermediate actions, inspect the optimized plan, then deliberately trigger computation with one display action.

In [ ]:
lazy_pipeline = (retail_orders
    .filter(F.col("order_status") == "COMPLETE")
    .select("store_id", "order_id", "total_amount")
    .groupBy("store_id")
    .agg(F.sum("total_amount").alias("revenue"))
    .filter(F.col("revenue") > 10000))

print("No action has run merely by defining lazy_pipeline.")
lazy_pipeline.explain("formatted")

# show() is the action that now asks Spark to compute the result.
lazy_pipeline.orderBy(F.desc("revenue")).show()

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

The plan includes pushed filters/projections plus an exchange for `groupBy`; stores S001, S002, and S004 exceed ₹10,000.

### Business interpretation

Spark can combine the filter and projection before shuffling only the fields required for store totals.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_orders` and describe each column used today.
2. [Beginner] Use lazy evaluation to answer a simple business question on `retail_orders`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine lazy evaluation with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use logical plan and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **tracing a lazy banking transaction pipeline from transformation to action**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Build a filter-select pipeline without an action.
2. Add a grouped total and predict the exchange.
3. Call `explain('formatted')`.
4. Label every operation as transformation or action.
5. Identify narrow transformations.
6. Identify the wide transformation.
7. Trigger exactly one bounded action.
8. Explain why repeated `count()` calls recompute by default.
9. Find pushed filters in a Parquet plan.
10. Draw the optimized DAG in a markdown cell.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

A notebook takes 40 minutes because analysts repeatedly call counts after every step. Redesign the validation strategy while retaining trustworthy checkpoints.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Expecting transformations to run immediately | Spark is lazy. | Identify the first action and inspect its full lineage. |
| Adding many debugging actions | Each action can recompute the pipeline. | Use targeted validations and persist only justified reused intermediates. |
| Reading only source code | The physical plan may differ after optimization. | Use formatted explain and Spark UI evidence. |
| Assuming every transformation shuffles | Many filters and projections are narrow. | Look for exchanges in the physical plan. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Spark Execution and Lazy Evaluation?**  
Lazy evaluation means Spark records transformations as a plan and computes them only when an action needs a result.

**2. Why do we need it?**  
Spark can optimize the full chain, avoid unused work, and pipeline compatible operations.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `explain`, `count`, `take`, `write`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Engineers use `explain`, Spark UI, event logs, and metrics to connect source code to actual execution.

**5. Can you give a simple analogy?**  
You write a complete shopping list before visiting the store, allowing one optimized trip instead of traveling after every item.

#### Intermediate

**1. What does Spark do internally?**  
Catalyst rewrites DataFrame plans, and exchanges mark shuffle boundaries that split stages in the DAG scheduler.

**2. What is the main performance consideration?**  
Inspect physical plans, minimize repeated actions, and persist only reused expensive plans.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare narrow and wide transformations?**  
Narrow operations consume a small known set of parent partitions; wide operations redistribute data and introduce shuffle boundaries.

**5. What common mistake would you watch for?**  
I watch for expecting transformations to run immediately. It happens because spark is lazy. I prevent it by identify the first action and inspect its full lineage.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Define a chain without intermediate actions, inspect the optimized plan, then deliberately trigger computation with one display action.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same spark execution and lazy evaluation principle, and validate tracing a lazy banking transaction pipeline from transformation to action without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Inspect physical plans, minimize repeated actions, and persist only reused expensive plans.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** lazy evaluation, logical plan, DAG, narrow transformation, wide transformation, action.
- **Important functions:** `explain`, `count`, `take`, `write`.
- **Retail problem solved:** build a lazy completed-sales pipeline, inspect its plan, and identify the action and shuffle boundary.
- **Banking work:** you designed and validated tracing a lazy banking transaction pipeline from transformation to action without receiving a copy-paste solution.
- **Interview takeaway:** Narrow operations consume a small known set of parent partitions; wide operations redistribute data and introduce shuffle boundaries.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 20 — Partitioning

**Project milestone:** Compare repartition and coalesce, then write completed orders partitioned by store ID.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain partitioning in plain language and say why a data engineer needs it.
- Recognize the core ideas: partition, repartition, coalesce, partitioned write, data skew.
- Use the main APIs safely: `repartition`, `coalesce`, `getNumPartitions`, `partitionBy`.
- Implement the retail requirement: compare repartition and coalesce, then write completed orders partitioned by store ID.
- Apply the same idea independently to partitioning transactions by branch or processing date while controlling output files.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Every stage launches tasks for its partitions. Exchanges create new partitioning, while coalesce can collapse existing dependencies.
- Answer interview questions about repartition and coalesce and production trade-offs.

## 2. Detailed Notes

### What is it?

A partition is a unit of distributed data processed by one task at a time; partitioning controls parallelism and file layout.

### Why do we need it?

Too few partitions underuse the cluster, too many add overhead, and uneven partitions create stragglers.

### How does it work?

`repartition` reshuffles to rebalance or partition by keys; `coalesce` usually reduces partitions with less movement; partitioned writes create directory layouts.

### What happens inside Spark?

Every stage launches tasks for its partitions. Exchanges create new partitioning, while coalesce can collapse existing dependencies.

### What happens in production?

Teams size files, choose low-cardinality partition columns, monitor skew, and separate compute partitioning from storage partitioning.

### Simple analogy

Partitions are boxes assigned to workers: giant boxes delay one worker, while thousands of tiny boxes waste handling time.

### Performance and design note

Target balanced, reasonably sized partitions and avoid high-cardinality directory partition columns.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `repartition` | Redistribute partitions | `df.repartition(8, 'store_id')` | Balance or key distribution |
| `coalesce` | Reduce partitions | `df.coalesce(2)` | Fewer output files |
| `getNumPartitions` | Inspect partition count | `df.rdd.getNumPartitions()` | Diagnostics |
| `partitionBy` | Directory-partition writes | `writer.partitionBy('state')` | Prunable storage layout |

## 4. Retail Dataset — Retail orders by store and date

Primary DataFrame for today: `retail_orders`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_orders.printSchema()
retail_orders.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Compare repartition and coalesce, then write completed orders partitioned by store ID.

**Plan before coding**

Filter first, explicitly rebalance by store, reduce for demonstration, and separately write a store-partitioned Parquet layout.

In [ ]:
completed = retail_orders.filter(F.col("order_status") == "COMPLETE")
by_store = completed.repartition(4, "store_id")
compact = by_store.coalesce(2)

print("Original partitions:", completed.rdd.getNumPartitions())
print("After repartition:", by_store.rdd.getNumPartitions())
print("After coalesce:", compact.rdd.getNumPartitions())

partition_path = f"{lab_root}/orders_by_store"
(completed.write.mode("overwrite")
    .partitionBy("store_id")
    .parquet(partition_path))
print("Partitioned output:", partition_path)

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

Repartition reports 4 partitions and coalesce reports 2; the output folder contains store-specific directories.

### Business interpretation

Compute parallelism and storage layout are related but distinct choices, each driven by workload access patterns.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_orders` and describe each column used today.
2. [Beginner] Use partition to answer a simple business question on `retail_orders`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine partition with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use repartition and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **partitioning transactions by branch or processing date while controlling output files**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Inspect transaction partition count.
2. Repartition transactions to four partitions.
3. Repartition by account ID.
4. Compare partition row counts.
5. Coalesce to two partitions.
6. Explain when coalesce can remain skewed.
7. Write by a low-cardinality branch or date field after integration.
8. Explain why customer ID is a poor directory partition.
9. Estimate file-count implications.
10. Inspect the plan for the repartition exchange.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

A daily bank table writes 30,000 tiny files and one branch contains 60% of transactions. Propose compute and storage partition strategies with measurable targets.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| One partition | A global operation or coalesce removed parallelism. | Increase balanced partitions before heavy downstream work. |
| Thousands of tiny files | Excessive partitions are written independently. | Compact and tune output partitions to target file sizes. |
| High-cardinality partitionBy | Directory count explodes. | Choose common filter columns with bounded cardinality. |
| Ignoring skew | One key owns a disproportionate share of rows. | Measure key frequency and apply a workload-specific skew strategy. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Partitioning?**  
A partition is a unit of distributed data processed by one task at a time; partitioning controls parallelism and file layout.

**2. Why do we need it?**  
Too few partitions underuse the cluster, too many add overhead, and uneven partitions create stragglers.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `repartition`, `coalesce`, `getNumPartitions`, `partitionBy`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Teams size files, choose low-cardinality partition columns, monitor skew, and separate compute partitioning from storage partitioning.

**5. Can you give a simple analogy?**  
Partitions are boxes assigned to workers: giant boxes delay one worker, while thousands of tiny boxes waste handling time.

#### Intermediate

**1. What does Spark do internally?**  
Every stage launches tasks for its partitions. Exchanges create new partitioning, while coalesce can collapse existing dependencies.

**2. What is the main performance consideration?**  
Target balanced, reasonably sized partitions and avoid high-cardinality directory partition columns.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare repartition and coalesce?**  
`repartition` can increase or decrease and performs a shuffle for balance; `coalesce` normally reduces with less movement but may preserve imbalance.

**5. What common mistake would you watch for?**  
I watch for one partition. It happens because a global operation or coalesce removed parallelism. I prevent it by increase balanced partitions before heavy downstream work.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Filter first, explicitly rebalance by store, reduce for demonstration, and separately write a store-partitioned Parquet layout.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same partitioning principle, and validate partitioning transactions by branch or processing date while controlling output files without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Target balanced, reasonably sized partitions and avoid high-cardinality directory partition columns.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** partition, repartition, coalesce, partitioned write, data skew.
- **Important functions:** `repartition`, `coalesce`, `getNumPartitions`, `partitionBy`.
- **Retail problem solved:** compare repartition and coalesce, then write completed orders partitioned by store ID.
- **Banking work:** you designed and validated partitioning transactions by branch or processing date while controlling output files without receiving a copy-paste solution.
- **Interview takeaway:** `repartition` can increase or decrease and performs a shuffle for balance; `coalesce` normally reduces with less movement but may preserve imbalance.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 21 — Performance Optimization

**Project milestone:** Optimize a reused completed-order dataset and join it to the small store dimension with a verified broadcast.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain performance optimization in plain language and say why a data engineer needs it.
- Recognize the core ideas: cache, persist, broadcast join, predicate pushdown, column pruning.
- Use the main APIs safely: `cache`, `persist`, `unpersist`, `broadcast`.
- Implement the retail requirement: optimize a reused completed-order dataset and join it to the small store dimension with a verified broadcast.
- Apply the same idea independently to optimizing customer-account joins with measurement-driven caching and broadcasting.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Catalyst prunes columns and pushes filters; the planner chooses joins; cached blocks let later actions skip upstream lineage.
- Answer interview questions about cache, persist, and checkpoint and production trade-offs.

## 2. Detailed Notes

### What is it?

Performance optimization reduces time and resources while preserving exactly the same business result.

### Why do we need it?

Efficient jobs meet SLAs, control cloud cost, and leave cluster capacity for other workloads.

### How does it work?

Engineers reduce data early, choose good formats and partitions, reuse expensive results selectively, and influence join strategies only with evidence.

### What happens inside Spark?

Catalyst prunes columns and pushes filters; the planner chooses joins; cached blocks let later actions skip upstream lineage.

### What happens in production?

Optimization begins with metrics and plans, changes one bottleneck at a time, and verifies both correctness and improvement.

### Simple analogy

Optimization is improving a delivery route after measuring traffic, not simply buying more trucks.

### Performance and design note

Measure before and after; cache only reused results, broadcast only bounded dimensions, and unpersist promptly.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `cache` | Persist with default level | `df.cache()` | Reused intermediate |
| `persist` | Choose storage level | `df.persist(StorageLevel.MEMORY_AND_DISK)` | Controlled reuse |
| `unpersist` | Release cached blocks | `df.unpersist()` | Resource cleanup |
| `broadcast` | Hint small-side broadcast | `F.broadcast(dim)` | Avoid large join shuffle |
| `explain` | Verify plan choices | `df.explain('formatted')` | Evidence |

## 4. Retail Dataset — Retail customer-order join

Primary DataFrame for today: `retail_orders`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_orders.printSchema()
retail_orders.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Optimize a reused completed-order dataset and join it to the small store dimension with a verified broadcast.

**Plan before coding**

Reduce rows and columns, persist a truly reused intermediate, broadcast the bounded dimension, verify the plan, and release memory.

In [ ]:
from pyspark import StorageLevel

reused_completed = (retail_orders
    .filter(F.col("order_status") == "COMPLETE")
    .select("order_id", "store_id", "customer_id", "order_date", "total_amount")
    .persist(StorageLevel.MEMORY_AND_DISK))

# Materialize once because two downstream consumers reuse it.
print("Completed rows:", reused_completed.count())

optimized_join = reused_completed.join(
    F.broadcast(retail_stores.select("store_id", "store_name", "state")),
    "store_id", "left"
)
optimized_join.explain("formatted")
optimized_join.groupBy("state").agg(F.sum("total_amount").alias("revenue")).show()

reused_completed.unpersist()

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

The plan contains `BroadcastHashJoin`; state revenue includes KA ₹100,648.50, MH ₹41,499.50, DL ₹7,601, and ONLINE ₹10,499.

### Business interpretation

The large fact remains distributed while each executor receives a small store lookup, avoiding a two-sided join shuffle.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_orders` and describe each column used today.
2. [Beginner] Use cache to answer a simple business question on `retail_orders`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine cache with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use persist and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **optimizing customer-account joins with measurement-driven caching and broadcasting**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Filter and project transaction facts before a join.
2. Broadcast the small branch dimension.
3. Verify `BroadcastHashJoin` in the plan.
4. Cache a reused successful-transaction intermediate.
5. Materialize the cache once.
6. Use it for two different summaries.
7. Unpersist after use.
8. Compare plans with and without broadcast.
9. Explain predicate pushdown in a Parquet read.
10. Write a before/after measurement checklist.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

A bank job regressed from 20 to 70 minutes. Build an evidence-first investigation covering input growth, plan changes, skew, shuffle, spill, file counts, and cache use.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Caching everything | Memory pressure and eviction outweigh recomputation savings. | Cache only expensive reused results and unpersist. |
| Broadcasting a large table | Executors may run out of memory. | Use statistics and size evidence before forcing broadcast. |
| Optimizing without a baseline | Improvement cannot be proven. | Record duration, input size, shuffle, spill, and output metrics. |
| Changing logic while tuning | Faster output may be incorrect. | Reconcile counts and totals before and after every optimization. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Performance Optimization?**  
Performance optimization reduces time and resources while preserving exactly the same business result.

**2. Why do we need it?**  
Efficient jobs meet SLAs, control cloud cost, and leave cluster capacity for other workloads.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `cache`, `persist`, `unpersist`, `broadcast`, `explain`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Optimization begins with metrics and plans, changes one bottleneck at a time, and verifies both correctness and improvement.

**5. Can you give a simple analogy?**  
Optimization is improving a delivery route after measuring traffic, not simply buying more trucks.

#### Intermediate

**1. What does Spark do internally?**  
Catalyst prunes columns and pushes filters; the planner chooses joins; cached blocks let later actions skip upstream lineage.

**2. What is the main performance consideration?**  
Measure before and after; cache only reused results, broadcast only bounded dimensions, and unpersist promptly.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare cache, persist, and checkpoint?**  
Cache uses the default persistence level, persist chooses a storage level, and checkpoint truncates lineage by writing reliable materialized data.

**5. What common mistake would you watch for?**  
I watch for caching everything. It happens because memory pressure and eviction outweigh recomputation savings. I prevent it by cache only expensive reused results and unpersist.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Reduce rows and columns, persist a truly reused intermediate, broadcast the bounded dimension, verify the plan, and release memory.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same performance optimization principle, and validate optimizing customer-account joins with measurement-driven caching and broadcasting without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Measure before and after; cache only reused results, broadcast only bounded dimensions, and unpersist promptly.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** cache, persist, broadcast join, predicate pushdown, column pruning.
- **Important functions:** `cache`, `persist`, `unpersist`, `broadcast`, `explain`.
- **Retail problem solved:** optimize a reused completed-order dataset and join it to the small store dimension with a verified broadcast.
- **Banking work:** you designed and validated optimizing customer-account joins with measurement-driven caching and broadcasting without receiving a copy-paste solution.
- **Interview takeaway:** Cache uses the default persistence level, persist chooses a storage level, and checkpoint truncates lineage by writing reliable materialized data.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 22 — Spark SQL

**Project milestone:** Calculate store revenue and average order value with Spark SQL, then compare the DataFrame plan.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain spark sql in plain language and say why a data engineer needs it.
- Recognize the core ideas: temporary view, SQL query, SQL/DataFrame equivalence, query plan.
- Use the main APIs safely: `createOrReplaceTempView`, `spark.sql`, `sql`, `explain`.
- Implement the retail requirement: calculate store revenue and average order value with Spark SQL, then compare the DataFrame plan.
- Apply the same idea independently to analyzing banking transactions with temporary views and auditable SQL.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: The SQL parser creates an unresolved plan, the analyzer resolves fields and functions, Catalyst optimizes it, and Spark executes a physical plan.
- Answer interview questions about Spark SQL and the DataFrame API and production trade-offs.

## 2. Detailed Notes

### What is it?

Spark SQL executes SQL queries over DataFrames and tables using the same Catalyst optimizer and execution engine.

### Why do we need it?

SQL makes analytics accessible and lets teams express joins, filters, aggregates, and windows declaratively.

### How does it work?

A DataFrame becomes queryable through a temporary view; `spark.sql` parses SQL into the same logical-plan system used by DataFrame APIs.

### What happens inside Spark?

The SQL parser creates an unresolved plan, the analyzer resolves fields and functions, Catalyst optimizes it, and Spark executes a physical plan.

### What happens in production?

Teams combine SQL models with PySpark orchestration, tests, reusable functions, and governed catalog tables.

### Simple analogy

DataFrame and SQL APIs are two languages giving instructions to the same kitchen.

### Performance and design note

Compare physical plans rather than assuming one API is faster; avoid opaque SQL strings and select only needed columns.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `createOrReplaceTempView` | Register session view | `df.createOrReplaceTempView('orders')` | SQL access |
| `spark.sql` | Execute SQL | `spark.sql('SELECT ...')` | Declarative transformations |
| `sql` | Use SQL expressions in select | `F.expr('CASE WHEN ... END')` | Mixed API code |
| `explain` | Inspect SQL plan | `result.explain('formatted')` | Plan comparison |

## 4. Retail Dataset — Retail sales temporary views

Primary DataFrame for today: `retail_orders`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_orders.printSchema()
retail_orders.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Calculate store revenue and average order value with Spark SQL, then compare the DataFrame plan.

**Plan before coding**

Register session-scoped views, express the same filtered join and aggregation in SQL, and inspect its physical plan.

In [ ]:
retail_orders.createOrReplaceTempView("orders")
retail_stores.createOrReplaceTempView("stores")

sql_store_kpis = spark.sql('''
SELECT
    s.store_id,
    s.store_name,
    COUNT(DISTINCT o.order_id) AS order_count,
    ROUND(SUM(o.total_amount), 2) AS revenue,
    ROUND(AVG(o.total_amount), 2) AS avg_order_value
FROM orders o
JOIN stores s ON o.store_id = s.store_id
WHERE o.order_status = 'COMPLETE'
GROUP BY s.store_id, s.store_name
ORDER BY revenue DESC
''')

sql_store_kpis.show(truncate=False)
sql_store_kpis.explain("formatted")

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

Bengaluru Central leads with **₹100,648.50** revenue; the plan contains join, filter, aggregate, and sort operations.

### Business interpretation

SQL users and PySpark developers can collaborate on one optimized execution engine and common business definitions.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_orders` and describe each column used today.
2. [Beginner] Use temporary view to answer a simple business question on `retail_orders`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine temporary view with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use SQL query and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **analyzing banking transactions with temporary views and auditable SQL**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Register customers, accounts, transactions, and branches as views.
2. Count successful transactions in SQL.
3. Calculate amount by transaction type.
4. Join transactions to accounts.
5. Join the result to branches.
6. Calculate branch/type KPIs.
7. Use a SQL CASE expression for amount bands.
8. Reproduce one query with the DataFrame API.
9. Compare physical plans.
10. Explain the scope and lifetime of a temporary view.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

Write a SQL design for monthly branch transaction KPIs with customer counts, debit/credit totals, and high-value flags, then list reconciliation checks.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| View not found | It was not registered in this session or the name differs. | Register the DataFrame and use consistent names. |
| Ambiguous SQL column | Joined tables share a field name. | Use table aliases and qualified references. |
| SQL injection through parameters | Untrusted text is concatenated into SQL. | Validate parameters or use safe DataFrame expressions. |
| Assuming SQL is automatically faster | Both APIs use the same optimizer. | Compare plans and readability, not syntax stereotypes. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Spark SQL?**  
Spark SQL executes SQL queries over DataFrames and tables using the same Catalyst optimizer and execution engine.

**2. Why do we need it?**  
SQL makes analytics accessible and lets teams express joins, filters, aggregates, and windows declaratively.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `createOrReplaceTempView`, `spark.sql`, `sql`, `explain`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Teams combine SQL models with PySpark orchestration, tests, reusable functions, and governed catalog tables.

**5. Can you give a simple analogy?**  
DataFrame and SQL APIs are two languages giving instructions to the same kitchen.

#### Intermediate

**1. What does Spark do internally?**  
The SQL parser creates an unresolved plan, the analyzer resolves fields and functions, Catalyst optimizes it, and Spark executes a physical plan.

**2. What is the main performance consideration?**  
Compare physical plans rather than assuming one API is faster; avoid opaque SQL strings and select only needed columns.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare Spark SQL and the DataFrame API?**  
Both normally reach the same optimizer and performance; the better choice is the clearest maintainable expression for the team and task.

**5. What common mistake would you watch for?**  
I watch for view not found. It happens because it was not registered in this session or the name differs. I prevent it by register the dataframe and use consistent names.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Register session-scoped views, express the same filtered join and aggregation in SQL, and inspect its physical plan.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same spark sql principle, and validate analyzing banking transactions with temporary views and auditable SQL without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Compare physical plans rather than assuming one API is faster; avoid opaque SQL strings and select only needed columns.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** temporary view, SQL query, SQL/DataFrame equivalence, query plan.
- **Important functions:** `createOrReplaceTempView`, `spark.sql`, `sql`, `explain`.
- **Retail problem solved:** calculate store revenue and average order value with Spark SQL, then compare the DataFrame plan.
- **Banking work:** you designed and validated analyzing banking transactions with temporary views and auditable SQL without receiving a copy-paste solution.
- **Interview takeaway:** Both normally reach the same optimizer and performance; the better choice is the clearest maintainable expression for the team and task.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 23 — Delta Lake

**Project milestone:** Create a Delta table, merge an updated order plus a new order, and read the original version.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain delta lake in plain language and say why a data engineer needs it.
- Recognize the core ideas: Delta table, ACID, transaction log, MERGE, update, delete, time travel.
- Use the main APIs safely: `format('delta')`, `DeltaTable.forPath`, `merge`, `update`.
- Implement the retail requirement: create a Delta table, merge an updated order plus a new order, and read the original version.
- Apply the same idea independently to building a banking transaction Delta table with idempotent upsert and audit history.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Optimistic concurrency validates a proposed commit against intervening versions before atomically publishing a new log entry.
- Answer interview questions about Parquet files and Delta tables and production trade-offs.

## 2. Detailed Notes

### What is it?

Delta Lake is a table format that adds a transaction log, ACID operations, schema controls, and versioned reads to data-lake files.

### Why do we need it?

Reliable pipelines need atomic commits, concurrent-read safety, upserts, deletes, and reproducible historical versions.

### How does it work?

Data files hold rows while `_delta_log` records table versions and committed actions; readers construct a consistent snapshot.

### What happens inside Spark?

Optimistic concurrency validates a proposed commit against intervening versions before atomically publishing a new log entry.

### What happens in production?

Bronze, Silver, and Gold tables use governed paths or catalog names, idempotent merges, retention policy, and access controls.

### Simple analogy

Parquet files are pages; Delta's transaction log is the official index recording exactly which pages belong to each edition.

### Performance and design note

Use sensible file sizes and partitioning, compact small files, collect statistics, and keep merge predicates selective.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `format('delta')` | Read/write Delta format | `df.write.format('delta').save(path)` | Delta table creation |
| `DeltaTable.forPath` | Open Delta table API | `DeltaTable.forPath(spark, path)` | DML operations |
| `merge` | Upsert source rows | `target.alias('t').merge(source.alias('s'), condition)` | Incremental loads |
| `update` | Update matched rows | `table.update(condition, values)` | Corrections |
| `delete` | Delete matched rows | `table.delete(condition)` | Governed removal |
| `versionAsOf` | Read historical version | `reader.option('versionAsOf', 0)` | Time travel |

### Optional Delta-enabled local session

Delta packages and configuration vary by Spark version. In Databricks, Delta is normally available already. Locally, install a compatible `delta-spark`, stop the existing session, restart the kernel, and create the session with `configure_spark_with_delta_pip` plus the Delta SQL extension and catalog configuration. The tutorial cell catches a missing setup so the rest of the course remains runnable.

## 4. Retail Dataset — Retail Delta orders table

Primary DataFrame for today: `retail_orders`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_orders.printSchema()
retail_orders.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Create a Delta table, merge an updated order plus a new order, and read the original version.

**Plan before coding**

Write the baseline atomically, merge by stable order ID, inspect changed rows, and use versioned read for reproducibility.

In [ ]:
# Delta requires a Delta-enabled SparkSession. Run the optional setup cell below,
# restart the kernel if packages are installed, then execute this cell.
delta_path = f"{lab_root}/delta/orders"

try:
    from delta.tables import DeltaTable

    retail_orders.write.format("delta").mode("overwrite").save(delta_path)
    updates = spark.createDataFrame([
        ("O1002", "C002", "S002", date(2026, 1, 5), "COMPLETE", 8500.0),
        ("O1016", "C003", "S001", date(2026, 3, 1), "COMPLETE", 1999.0),
    ], retail_orders.schema)

    target = DeltaTable.forPath(spark, delta_path)
    (target.alias("t")
        .merge(updates.alias("s"), "t.order_id = s.order_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

    spark.read.format("delta").load(delta_path).filter("order_id IN ('O1002','O1016')").show()
    print("Version 0 rows:", spark.read.format("delta").option("versionAsOf", 0).load(delta_path).count())
except Exception as exc:
    print("Delta demo not active in this kernel.")
    print("Follow the Delta setup note, restart the kernel, and rerun. Details:", str(exc)[:300])

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

With Delta enabled, `O1002` becomes COMPLETE, `O1016` is inserted, the current table has 16 rows, and version 0 has 15 rows.

### Business interpretation

The incremental load is idempotent by order ID and historical state remains queryable for audit and recovery.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_orders` and describe each column used today.
2. [Beginner] Use Delta table to answer a simple business question on `retail_orders`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine Delta table with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use ACID and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **building a banking transaction Delta table with idempotent upsert and audit history**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Create a Delta path for transactions.
2. Write the baseline table.
3. Create one corrected and one new transaction.
4. Merge by transaction ID.
5. Prove a retry does not add a duplicate.
6. Read version zero.
7. Describe an approved correction with update.
8. Describe a governed deletion use case.
9. Inspect table history if available.
10. List retention and privacy controls required by the bank.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

Design an idempotent transaction CDC merge with insert, correction, and reversal events, including ordering, audit columns, and late-arrival rules.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Delta data source not found | The package or session extensions are missing. | Install a compatible Delta package and restart with a Delta-enabled session. |
| Merge matches multiple source rows | Source keys are duplicated. | Deterministically deduplicate source events before merge. |
| Non-idempotent insert logic | Retry creates new business rows. | Match on stable keys and test reruns. |
| Unsafe vacuum/retention | Files needed for history may be removed. | Follow platform retention and governance policies. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Delta Lake?**  
Delta Lake is a table format that adds a transaction log, ACID operations, schema controls, and versioned reads to data-lake files.

**2. Why do we need it?**  
Reliable pipelines need atomic commits, concurrent-read safety, upserts, deletes, and reproducible historical versions.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `format('delta')`, `DeltaTable.forPath`, `merge`, `update`, `delete`, `versionAsOf`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Bronze, Silver, and Gold tables use governed paths or catalog names, idempotent merges, retention policy, and access controls.

**5. Can you give a simple analogy?**  
Parquet files are pages; Delta's transaction log is the official index recording exactly which pages belong to each edition.

#### Intermediate

**1. What does Spark do internally?**  
Optimistic concurrency validates a proposed commit against intervening versions before atomically publishing a new log entry.

**2. What is the main performance consideration?**  
Use sensible file sizes and partitioning, compact small files, collect statistics, and keep merge predicates selective.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare Parquet files and Delta tables?**  
Parquet defines a columnar file format; Delta adds a transaction log and table semantics while storing data in Parquet files.

**5. What common mistake would you watch for?**  
I watch for delta data source not found. It happens because the package or session extensions are missing. I prevent it by install a compatible delta package and restart with a delta-enabled session.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Write the baseline atomically, merge by stable order ID, inspect changed rows, and use versioned read for reproducibility.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same delta lake principle, and validate building a banking transaction Delta table with idempotent upsert and audit history without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Use sensible file sizes and partitioning, compact small files, collect statistics, and keep merge predicates selective.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** Delta table, ACID, transaction log, MERGE, update, delete, time travel.
- **Important functions:** `format('delta')`, `DeltaTable.forPath`, `merge`, `update`, `delete`, `versionAsOf`.
- **Retail problem solved:** create a Delta table, merge an updated order plus a new order, and read the original version.
- **Banking work:** you designed and validated building a banking transaction Delta table with idempotent upsert and audit history without receiving a copy-paste solution.
- **Interview takeaway:** Parquet defines a columnar file format; Delta adds a transaction log and table semantics while storing data in Parquet files.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 24 — End-to-End ETL: Bronze, Silver, Gold

**Project milestone:** Build reproducible Bronze metadata, validated Silver order lines, and a Gold daily-store-category sales table.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain end-to-end etl: bronze, silver, gold in plain language and say why a data engineer needs it.
- Recognize the core ideas: Bronze, Silver, Gold, data quality, idempotency, lineage.
- Use the main APIs safely: `write.mode`, `withColumn`, `join`, `groupBy / agg`.
- Implement the retail requirement: build reproducible Bronze metadata, validated Silver order lines, and a Gold daily-store-category sales table.
- Apply the same idea independently to designing the parallel Banking Transaction Analytics Platform across Bronze, Silver, and Gold.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: Each layer materializes a contract, reducing repeated lineage and providing restart points; the storage format controls atomicity and versioning.
- Answer interview questions about Bronze, Silver, and Gold and production trade-offs.

## 2. Detailed Notes

### What is it?

Medallion ETL organizes raw ingestion, cleaned integrated data, and business-ready outputs into Bronze, Silver, and Gold layers.

### Why do we need it?

Layered responsibilities make pipelines testable, recoverable, auditable, and easier for teams to own.

### How does it work?

Bronze preserves source fidelity and metadata; Silver enforces contracts and integrates entities; Gold publishes stable business grains and KPIs.

### What happens inside Spark?

Each layer materializes a contract, reducing repeated lineage and providing restart points; the storage format controls atomicity and versioning.

### What happens in production?

Orchestration supplies run IDs and dates, quality gates block bad promotion, and monitoring records counts, totals, duration, and freshness.

### Simple analogy

Bronze is the receiving dock, Silver is the inspected warehouse, and Gold is the finished-product showroom.

### Performance and design note

Incrementally process changed partitions, compact files, reuse conformed dimensions, and design Gold tables for actual access patterns.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `write.mode` | Choose write behavior | `df.write.mode('overwrite')` | Layer materialization |
| `withColumn` | Add lineage/quality fields | `df.withColumn('ingest_date', F.current_date())` | Bronze metadata |
| `join` | Integrate conformed entities | `facts.join(dim, key)` | Silver models |
| `groupBy / agg` | Publish measures | `df.groupBy(dim).agg(...) ` | Gold KPIs |
| `partitionBy` | Lay out incremental data | `writer.partitionBy('business_date')` | Prunable storage |

## 4. Retail Dataset — Complete retail project datasets

Primary DataFrame for today: `retail_orders`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_orders.printSchema()
retail_orders.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Build reproducible Bronze metadata, validated Silver order lines, and a Gold daily-store-category sales table.

**Plan before coding**

Give each layer one responsibility, carry run lineage, integrate only validated facts, publish a declared Gold grain, and enforce quality checks before promotion.

In [ ]:
run_id = "classroom-run-2026-03-01"

# Bronze: source fidelity plus ingestion metadata.
bronze_orders = (retail_orders
    .withColumn("source_system", F.lit("retail_pos"))
    .withColumn("ingest_run_id", F.lit(run_id)))

# Silver: valid completed orders integrated with line and product context.
silver_sales = (bronze_orders
    .filter((F.col("order_status") == "COMPLETE") & F.col("order_id").isNotNull())
    .join(retail_order_items, "order_id", "inner")
    .join(retail_products.select("product_id", "product_name", "category"), "product_id", "left")
    .withColumn("net_line_amount",
                F.round(F.col("quantity") * F.col("unit_price") * (1 - F.col("discount")), 2))
    .select("order_id", "order_date", "store_id", "customer_id", "product_id",
            "product_name", "category", "quantity", "net_line_amount", "ingest_run_id"))

# Gold grain: one row per order date, store, and category.
gold_daily_sales = (silver_sales
    .groupBy("order_date", "store_id", "category")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("net_line_amount"), 2).alias("revenue"),
    ))

assert bronze_orders.count() == retail_orders.count()
assert silver_sales.filter(F.col("category").isNull()).count() == 0
gold_daily_sales.orderBy("order_date", "store_id", "category").show(50, truncate=False)

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

Bronze retains 15 order rows; Silver has completed line-level sales with no null category; Gold returns daily store-category KPI rows.

### Business interpretation

The project now separates reproducible raw history, reusable clean detail, and dashboard-ready business metrics.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_orders` and describe each column used today.
2. [Beginner] Use Bronze to answer a simple business question on `retail_orders`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine Bronze with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use Silver and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **designing the parallel Banking Transaction Analytics Platform across Bronze, Silver, and Gold**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Define Bronze tables and lineage columns for all six banking datasets.
2. Write schema and required-key checks.
3. Create accepted and rejected transaction outputs.
4. Build a Silver customer-account view.
5. Build a Silver transaction fact with branch and customer keys.
6. Create transaction amount and high-value flags.
7. Define a daily branch/type Gold grain.
8. Calculate count, total, average, and distinct customers.
9. Add reconciliation from raw to Silver to Gold.
10. Document incremental keys and rerun behavior.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

Design the complete banking pipeline for late events, duplicate transactions, customer privacy, audit retention, quality gates, and daily KPI publication.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| Bronze data is cleaned destructively | Raw source fidelity is lost. | Preserve original values and add metadata in Bronze. |
| Silver is dashboard-specific | Reusable entities become coupled to one report. | Keep conformed detail reusable; put report logic in Gold. |
| Gold grain is undocumented | Consumers double-count measures. | State keys and one-row-per definition in the contract. |
| No rerun design | Retries duplicate or overwrite incorrect scope. | Use idempotent keys, partitions, merges, and run audit. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is End-to-End ETL: Bronze, Silver, Gold?**  
Medallion ETL organizes raw ingestion, cleaned integrated data, and business-ready outputs into Bronze, Silver, and Gold layers.

**2. Why do we need it?**  
Layered responsibilities make pipelines testable, recoverable, auditable, and easier for teams to own.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `write.mode`, `withColumn`, `join`, `groupBy / agg`, `partitionBy`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
Orchestration supplies run IDs and dates, quality gates block bad promotion, and monitoring records counts, totals, duration, and freshness.

**5. Can you give a simple analogy?**  
Bronze is the receiving dock, Silver is the inspected warehouse, and Gold is the finished-product showroom.

#### Intermediate

**1. What does Spark do internally?**  
Each layer materializes a contract, reducing repeated lineage and providing restart points; the storage format controls atomicity and versioning.

**2. What is the main performance consideration?**  
Incrementally process changed partitions, compact files, reuse conformed dimensions, and design Gold tables for actual access patterns.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare Bronze, Silver, and Gold?**  
Bronze preserves raw truth, Silver creates clean reusable entities, and Gold serves a defined business question at a stable grain.

**5. What common mistake would you watch for?**  
I watch for bronze data is cleaned destructively. It happens because raw source fidelity is lost. I prevent it by preserve original values and add metadata in bronze.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Give each layer one responsibility, carry run lineage, integrate only validated facts, publish a declared Gold grain, and enforce quality checks before promotion.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same end-to-end etl: bronze, silver, gold principle, and validate designing the parallel Banking Transaction Analytics Platform across Bronze, Silver, and Gold without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Incrementally process changed partitions, compact files, reuse conformed dimensions, and design Gold tables for actual access patterns.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** Bronze, Silver, Gold, data quality, idempotency, lineage.
- **Important functions:** `write.mode`, `withColumn`, `join`, `groupBy / agg`, `partitionBy`.
- **Retail problem solved:** build reproducible Bronze metadata, validated Silver order lines, and a Gold daily-store-category sales table.
- **Banking work:** you designed and validated designing the parallel Banking Transaction Analytics Platform across Bronze, Silver, and Gold without receiving a copy-paste solution.
- **Interview takeaway:** Bronze preserves raw truth, Silver creates clean reusable entities, and Gold serves a defined business question at a stable grain.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Day 25 — Capstone Project, Debugging, and Interview Readiness

**Project milestone:** Publish the final retail KPI pack: revenue, time trends, store/category performance, top products/customers, AOV, and purchase frequency.

## 1. Learning Objectives

By the end of this session, you should be able to:

- Explain capstone project, debugging, and interview readiness in plain language and say why a data engineer needs it.
- Recognize the core ideas: KPI contract, end-to-end project, debugging, optimization, interview storytelling.
- Use the main APIs safely: `assert`, `explain`, `join`, `groupBy / agg`.
- Implement the retail requirement: publish the final retail KPI pack: revenue, time trends, store/category performance, top products/customers, AOV, and purchase frequency.
- Apply the same idea independently to completing the Banking Transaction Analytics Platform and presenting it as an interview project.
- Predict the schema and result before running a transformation.
- Describe Spark's internal behavior: The final Spark application becomes a DAG of scans, projections, filters, joins, exchanges, aggregates, windows, and writes.
- Answer interview questions about a notebook demo and a production data product and production trade-offs.

## 2. Detailed Notes

### What is it?

The capstone combines data modeling, transformations, validation, performance reasoning, and communication into one production-style solution.

### Why do we need it?

Real data engineering work is not a list of APIs; it is delivering trustworthy, maintainable data products and explaining design choices.

### How does it work?

Start from requirements and grain, build layered transformations, validate every boundary, inspect execution, and publish documented KPIs.

### What happens inside Spark?

The final Spark application becomes a DAG of scans, projections, filters, joins, exchanges, aggregates, windows, and writes.

### What happens in production?

A release includes code review, tests, job configuration, orchestration, monitoring, ownership, runbook, lineage, and access controls.

### Simple analogy

The capstone is the final building inspection: structure, utilities, safety checks, and user needs must all work together.

### Performance and design note

Optimize the measured critical path only after correctness; verify plans, skew, shuffle, spill, file sizes, and workload reuse.

## 3. Important PySpark Functions

| Function | Purpose | Syntax | Common use |
|---|---|---|---|
| `assert` | Fail fast on invariants | `assert bad_rows == 0` | Data-quality gate |
| `explain` | Inspect execution plan | `df.explain('formatted')` | Performance review |
| `join` | Integrate entities | `facts.join(dim, key)` | Project modeling |
| `groupBy / agg` | Build KPIs | `df.groupBy(...).agg(...) ` | Gold outputs |
| `Window` | Add rankings/sequences | `function.over(window)` | Advanced analytics |

## 4. Retail Dataset — Retail Sales Analytics Platform capstone

Primary DataFrame for today: `retail_orders`.

Run the inspection cell before the tutorial. `printSchema()` shows names, data types, and nullability; `show()` displays records but does not guarantee business order unless `orderBy` is used.

In [ ]:
retail_orders.printSchema()
retail_orders.show(20, truncate=False)

## 5. Retail Hands-On Tutorial

**Business requirement**

Publish the final retail KPI pack: revenue, time trends, store/category performance, top products/customers, AOV, and purchase frequency.

**Plan before coding**

Reuse validated Silver detail, create one DataFrame per KPI contract, rank products deterministically, and finish with invariants and explain-plan review.

In [ ]:
# Reuse the validated Silver sales detail from Day 24.
retail_kpis = {
    "total_revenue": silver_sales.agg(F.round(F.sum("net_line_amount"), 2).alias("total_revenue")),
    "daily_revenue": silver_sales.groupBy("order_date").agg(F.round(F.sum("net_line_amount"), 2).alias("revenue")),
    "monthly_revenue": (silver_sales
        .withColumn("month", F.date_format("order_date", "yyyy-MM"))
        .groupBy("month").agg(F.round(F.sum("net_line_amount"), 2).alias("revenue"))),
    "store_revenue": silver_sales.groupBy("store_id").agg(F.round(F.sum("net_line_amount"), 2).alias("revenue")),
    "category_revenue": silver_sales.groupBy("category").agg(F.round(F.sum("net_line_amount"), 2).alias("revenue")),
    "customer_metrics": (silver_sales.groupBy("customer_id")
        .agg(F.countDistinct("order_id").alias("purchase_frequency"),
             F.round(F.sum("net_line_amount"), 2).alias("customer_revenue"))),
}

product_rank_window = Window.orderBy(F.desc("product_revenue"), F.asc("product_id"))
retail_kpis["top_products"] = (silver_sales.groupBy("product_id", "product_name")
    .agg(F.round(F.sum("net_line_amount"), 2).alias("product_revenue"))
    .withColumn("sales_rank", F.dense_rank().over(product_rank_window)))

completed_order_values = (retail_orders.filter("order_status = 'COMPLETE'")
    .agg(F.round(F.avg("total_amount"), 2).alias("average_order_value")))
retail_kpis["average_order_value"] = completed_order_values

for name, result in retail_kpis.items():
    print(f"\n=== {name} ===")
    result.orderBy(result.columns[0]).show(20, truncate=False) if len(result.columns) > 1 else result.show()

assert retail_orders.select("order_id").distinct().count() == retail_orders.count()
assert silver_sales.filter(F.col("net_line_amount") < 0).count() == 0

### Code explanation

- The first operation establishes the correct source and business population.
- Column expressions remain distributed; Spark builds a plan instead of looping through rows in Python.
- Names and output grain are made explicit so downstream users know what one row represents.
- The final display is intentionally small; production pipelines normally validate and write the distributed result.

### Expected output

The notebook displays all required KPI tables; average completed order value is **₹12,326.77**, with product and customer leaders ranked from validated detail.

### Business interpretation

The 25-day project delivers interview-ready evidence: code, business grains, controls, architecture reasoning, and performance awareness.

## 6. Retail Practice Problems

1. [Beginner] Display five rows from `retail_orders` and describe each column used today.
2. [Beginner] Use KPI contract to answer a simple business question on `retail_orders`.
3. [Beginner] Re-create the instructor result with one changed value or condition.
4. [Intermediate] Combine KPI contract with a column selection learned earlier and give columns business-friendly aliases.
5. [Intermediate] Use end-to-end project and verify the output schema before accepting the result.
6. [Intermediate] Add a data-quality check for null, duplicate, or invalid values relevant to today's result.
7. [Intermediate] Produce a second result at a different business grain and explain why its row count changes.
8. [Advanced] Solve the retail requirement in a second valid way and compare readability and execution plans.
9. [Advanced] Explain how today's transformation behaves with 500 million rows and one highly skewed value.
10. [Advanced] Turn today's work into an idempotent pipeline step with named inputs, output columns, and validation rules.

## 7. Banking Assignment

            **Dataset/schema**

            Use the notebook tables most relevant to this session: `bank_customers`, `bank_accounts`, `bank_transactions`, `bank_branches`, `bank_loans`, and `bank_credit_cards`. Inspect the exact schema before coding and use stable IDs as keys.

            **Business scenario**

            Your banking team needs a reliable result for **completing the Banking Transaction Analytics Platform and presenting it as an interview project**. Use today's concepts plus only previously taught concepts.

            **Assignment questions**

            1. Build total successful transaction amount.
2. Compare debit and credit count/amount.
3. Calculate average transaction amount.
4. Calculate transactions per customer.
5. Calculate branch transaction volume.
6. Rank top customers by activity.
7. Publish account balance KPIs without duplicating balances.
8. Calculate loan exposure by type/status.
9. Identify high-value transactions with a parameter.
10. Design suspicious-pattern flags and evidence output.

            **Expected result requirements**

            - State the output grain (one row per what) before the code.
            - Return business-friendly column names and deterministic presentation order where needed.
            - Include record-count, key, null, and financial-total checks appropriate to the result.
            - Do not use `collect()` for the full dataset or use a concept scheduled for a later day.
            - Explain the business meaning in 3–5 sentences; screenshots alone are not a submission.

            > A complete solution is intentionally not included. Build it in your answer notebook, compare it with the stated requirements, and ask for review only after recording your own reasoning.

## 8. Banking Challenge

Present a 10-minute banking capstone walkthrough covering requirements, architecture, grain, quality, joins, windows, optimization evidence, security, failure recovery, and three future improvements.

**Deliverables:** a stated grain, transformation plan, PySpark implementation, expected evidence columns, two quality checks, and one scale/security consideration.

## 9. Common Errors

| Error / problem | Why it happens | How to fix it |
|---|---|---|
| KPI definitions are implicit | Different teams calculate different populations or grains. | Publish formula, filters, grain, owner, and freshness. |
| Demo-only code is called production-ready | Tests, operations, security, and reruns are missing. | Describe the hardening gap honestly and add a delivery checklist. |
| Debugging starts with random code changes | The first failing boundary is unknown. | Reproduce, isolate, inspect schema/counts/plan, then change one cause. |
| Interview answer lists APIs only | It does not show engineering judgment. | Use situation, requirement, design, trade-off, validation, and result. |

## 10. Interview Questions and Short Spoken Answers

#### Beginner

**1. What is Capstone Project, Debugging, and Interview Readiness?**  
The capstone combines data modeling, transformations, validation, performance reasoning, and communication into one production-style solution.

**2. Why do we need it?**  
Real data engineering work is not a list of APIs; it is delivering trustworthy, maintainable data products and explaining design choices.

**3. Which PySpark functions are most relevant?**  
The main APIs here are `assert`, `explain`, `join`, `groupBy / agg`, `Window`. I choose among them based on the required output grain and schema.

**4. Where would you use this in a real project?**  
A release includes code review, tests, job configuration, orchestration, monitoring, ownership, runbook, lineage, and access controls.

**5. Can you give a simple analogy?**  
The capstone is the final building inspection: structure, utilities, safety checks, and user needs must all work together.

#### Intermediate

**1. What does Spark do internally?**  
The final Spark application becomes a DAG of scans, projections, filters, joins, exchanges, aggregates, windows, and writes.

**2. What is the main performance consideration?**  
Optimize the measured critical path only after correctness; verify plans, skew, shuffle, spill, file sizes, and workload reuse.

**3. How do you validate the result?**  
I verify schema, row grain, key uniqueness, null rates, record counts, and a small set of known business totals.

**4. How do you compare a notebook demo and a production data product?**  
A demo proves logic on a sample; a production product adds contracts, tests, idempotency, scale evidence, security, observability, and operational ownership.

**5. What common mistake would you watch for?**  
I watch for kpi definitions are implicit. It happens because different teams calculate different populations or grains. I prevent it by publish formula, filters, grain, owner, and freshness.

#### Scenario-based

**1. How would you solve today's retail requirement?**  
Reuse validated Silver detail, create one DataFrame per KPI contract, rank products deterministically, and finish with invariants and explain-plan review.

**2. How would you adapt it to banking data?**  
I would start from the declared banking keys, apply the same capstone project, debugging, and interview readiness principle, and validate completing the Banking Transaction Analytics Platform and presenting it as an interview project without copying retail-specific assumptions.

**3. What changes when the table grows to hundreds of millions of rows?**  
Optimize the measured critical path only after correctness; verify plans, skew, shuffle, spill, file sizes, and workload reuse.

**4. How would you productionize this notebook code?**  
I would move transformations into tested functions, parameterize paths and dates, add data-quality checks, log counts and metrics, and write atomically to a governed target.

**5. The result is unexpectedly wrong. How do you debug it?**  
I check the schema and grain at every step, isolate the earliest step where counts or totals diverge, inspect the physical plan, and reproduce the issue with a minimal sample.

## 11. Day Summary

- **Concepts learned:** KPI contract, end-to-end project, debugging, optimization, interview storytelling.
- **Important functions:** `assert`, `explain`, `join`, `groupBy / agg`, `Window`.
- **Retail problem solved:** publish the final retail KPI pack: revenue, time trends, store/category performance, top products/customers, AOV, and purchase frequency.
- **Banking work:** you designed and validated completing the Banking Transaction Analytics Platform and presenting it as an interview project without receiving a copy-paste solution.
- **Interview takeaway:** A demo proves logic on a sample; a production product adds contracts, tests, idempotency, scale evidence, security, observability, and operational ownership.

**Exit ticket:** Explain today's output grain, point to the first shuffle or action if one exists, and name one production validation you would add.

---

# Final Project Acceptance Checklist

## Retail Sales Analytics Platform

- [ ] Raw customers, products, orders, order items, stores, payments, and inventory are represented.
- [ ] Bronze retains source values and ingestion lineage.
- [ ] Silver has enforced keys/types, quality exceptions, and reusable integrated detail.
- [ ] Gold publishes total/daily/monthly revenue, revenue by store/category, top products/customers, AOV, purchase frequency, product ranking, and store performance.
- [ ] Counts and totals reconcile between layers.
- [ ] The physical plan and partition/file strategy have been reviewed.
- [ ] The pipeline is idempotent and has a rerun/recovery design.

## Banking Transaction Analytics Platform

- [ ] Raw customers, accounts, transactions, branches, loans, and credit cards are represented.
- [ ] Bronze, rejected-data, Silver, and Gold contracts are documented.
- [ ] Customer-account integration and transaction fact grains are correct.
- [ ] Gold includes total transaction amount, debit vs credit, average transaction amount, transactions/customer, branch volume, top customers, balances, loan exposure, high-value activity, and suspicious-pattern evidence.
- [ ] Financial measures are not duplicated by joins.
- [ ] Privacy, access, retention, and audit requirements are addressed.
- [ ] A 10-minute interview walkthrough and a one-page architecture diagram are ready.

# 25-Day Outcome

After completing both projects, you can independently build PySpark DataFrame transformations; read/write CSV, JSON, and Parquet; filter, join, aggregate, and use windows; write Spark SQL; reason about the DAG, tasks, partitions, shuffles, cache, and broadcast joins; explain Delta Lake and medallion architecture; debug common failures; and communicate design trade-offs confidently in interviews.

**Next practice cycle:** rerun the banking capstone from a blank notebook, add automated tests for five business rules, and explain every physical-plan exchange you see.